# PyTorch Fundamentals: From Tensors to a Trained Neural Network
### Notebook 1 of 2 · A self-study course for engineering undergraduates

---

You already know **what** a neural network does. You can describe a forward pass, compute a loss, explain gradient descent, derive backpropagation for a small network, and tell overfitting from underfitting. This notebook teaches you **how to write all of that in PyTorch**, the library most research papers and much of industry use.

By the end of this notebook you will be able to:

- create and manipulate tensors (PyTorch's arrays) without shape bugs;
- explain exactly what `loss.backward()` computes, and why `optimizer.zero_grad()` exists;
- define models with `nn.Module`, choose the right loss function, and feed data with `Dataset` and `DataLoader`;
- write a complete, honest training and evaluation pipeline, and use it to train a multi-layer perceptron (MLP) on **Fashion-MNIST** to roughly 89% test accuracy;
- read typical PyTorch code written by other people and predict what it does.

**Notebook 2** then covers the tools professionals use every day: strides and memory layout, `einsum`, learning-rate schedules, early stopping, checkpointing, weight averaging, convolutional networks, fine-tuning pretrained models, mixed precision, debugging, and exporting models.

## Contents and time budget

| Chapter | Topic | You will be able to… | Time |
| --- | --- | --- | --- |
| 0 | Setup and the big picture | Run a complete training script and know where each line is explained | 25 min |
| 1 | Tensors | Create, reshape, index, broadcast, reduce, combine and select tensors; move them to the GPU | 95 min |
| 2 | Autograd | Explain and use automatic differentiation | 60 min |
| 3 | Linear regression four ways | See exactly what each PyTorch abstraction replaces | 40 min |
| 4 | `nn.Module` | Write, inspect and save models | 60 min |
| 5 | Loss functions | Pick the right loss, output shape and target format | 40 min |
| 6 | Data | Build `Dataset`s and `DataLoader`s without data leakage | 50 min |
| 7 | Project: an MLP on Fashion-MNIST | Train, select and evaluate a model end to end | 75 min |
| | | **Total** | **≈ 7.5 h** |

The times are estimates for a first pass, including the exercises; tell us if yours differ a lot. GPU time is small: everything here also runs on a laptop CPU, just more slowly in Chapter 7.

---
# Chapter 0. Setup and how to study

### 0.1 Running the notebook
**On Google Colab (recommended):** open the notebook, then choose *Runtime → Change runtime type → T4 GPU → Save*. Run the setup cell below. It should print `device: cuda`.
**On your own machine:** install PyTorch from [pytorch.org](https://pytorch.org/get-started/locally/) (pick your OS and whether you have an NVIDIA GPU), plus `torchvision` and `matplotlib`. Without a GPU the notebook prints `device: cpu` and still works.

### 0.2 Conventions used throughout

| Symbol | Meaning |
| --- | --- |
| 🎯 | What you will be able to do after the chapter |
| 🤔 **Predict** | Stop and predict the output *before* running the next cell. Being wrong and finding out why is how you learn fastest. |
| ⚠️ **Pitfall** | A mistake real engineers make. Many of them produce **no error message**, which is what makes them dangerous. |
| 🏭 **Industry note** | A practice that matters more at work than in a course |
| 📖 **Code-reading drill** | A snippet in the style of real repositories. Read it and answer the questions. |
| 🧪 **Exercise** | An empty cell for your attempt, sometimes a 🔎 self-check cell that tests your function, then a hidden ✅ solution |
| ★☆☆ / ★★☆ / ★★★ | Exercise difficulty |

**Solutions are hidden.** In Colab a solution appears as a titled cell with its code hidden: double-click the cell to see the code. In JupyterLab, click the collapsed input. (Other viewers, such as VS Code or GitHub, show the code openly, so avoid peeking there.) Solutions do **not** run during *Run all* unless you set `RUN_SOLUTIONS = True` in the setup cell. Try first. A solution read after a real attempt sticks; a solution read instead of an attempt does not.

### 0.3 Notebook hygiene
A notebook remembers every variable from every cell you have run, in the order you ran them. To avoid confusing results:
1. Run cells **from top to bottom**.
2. Short names like `x`, `t` and `m` are scratch variables that get reused. Do not rely on one surviving from a distant cell.
3. When something behaves strangely, use *Runtime → Restart session and run all* (Colab) or *Kernel → Restart & Run All* (Jupyter).

In [ ]:
# ---- Setup: run this cell first ------------------------------------------------
import math, random, time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn                  # layers, models, losses  (convention: "nn")
import torch.nn.functional as F        # the same operations as plain functions (convention: "F")
import torchvision                     # datasets and image tools
import matplotlib.pyplot as plt

FAST_DEV_RUN = False   # True = tiny data and 1 epoch everywhere: a quick smoke test of the whole notebook
RUN_SOLUTIONS = False  # True = also execute the hidden exercise solutions during "Run all" (slower)
DATA_DIR = Path("./data")
DATA_DIR.mkdir(exist_ok=True)

# Use the best available hardware: an NVIDIA GPU ("cuda"), an Apple-silicon GPU ("mps"), or the CPU.
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

def set_seed(seed: int = 0) -> None:
    # Make random numbers repeatable: Python, NumPy and PyTorch each have their own generator.
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)            # also seeds every GPU

def n_epochs(full: int) -> int:
    # Number of training epochs: `full` normally, 1 in smoke-test mode.
    return 1 if FAST_DEV_RUN else full

def maybe_subset(ds, n: int = 512):
    # The whole dataset normally; only its first n items in smoke-test mode.
    return torch.utils.data.Subset(ds, range(min(n, len(ds)))) if FAST_DEV_RUN else ds

RESULTS = {}           # the numbers YOU measure, collected as you go and summarised at the end
def record(name, value):
    RESULTS[name] = value
    print(f"📝 recorded  {name}: {value:.4f}")

def self_check(name, test):
    # Test the function YOU defined under `name` and report pass/fail without revealing the solution.
    fn = globals().get(name)
    if fn is None:
        print(f"ℹ️  Define `{name}` in the cell above first, then run this cell again.")
        return
    try:
        test(fn)
        print(f"✅ `{name}` passed all checks.")
    except AssertionError as e:
        print(f"❌ `{name}` failed a check: {e}")
    except Exception as e:
        print(f"❌ `{name}` raised {type(e).__name__}: {e}")

print(f"torch {torch.__version__} | torchvision {torchvision.__version__} | device: {device}")
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
set_seed(0)

### 0.4 The big picture first: a complete training script in 20 lines

Before taking PyTorch apart piece by piece, look at the whole machine running. The cell below trains a small MLP to separate two interleaving half-moons of 2-D points. Every line maps onto something you already know from your neural-network course.

Do not worry about the details yet. The comments say which chapter explains each line. **Read the code first**, then run it.

In [ ]:
def make_moons(n=1000, noise=0.15, seed=0):
    # Two interleaving half-circles: a classic toy problem that a linear model cannot solve.
    g = torch.Generator().manual_seed(seed)
    n1 = n // 2
    t1, t2 = torch.rand(n1, generator=g) * math.pi, torch.rand(n - n1, generator=g) * math.pi
    upper = torch.stack([torch.cos(t1), torch.sin(t1)], dim=1)
    lower = torch.stack([1 - torch.cos(t2), 0.5 - torch.sin(t2)], dim=1)
    X = torch.cat([upper, lower]) + noise * torch.randn(n, 2, generator=g)
    y = torch.cat([torch.zeros(n1), torch.ones(n - n1)]).long()
    return X, y

set_seed(0)
X, y = make_moons()                                # X: (1000, 2) inputs, y: (1000,) labels   -> Ch. 1
model = nn.Sequential(                             # an MLP: 2 -> 32 -> 32 -> 2               -> Ch. 4
    nn.Linear(2, 32), nn.ReLU(),
    nn.Linear(32, 32), nn.ReLU(),
    nn.Linear(32, 2),                              # 2 outputs = one score ("logit") per class
)
loss_fn = nn.CrossEntropyLoss()                    # softmax + negative log-likelihood          -> Ch. 5
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)   # the update rule                  -> Ch. 3

for step in range(300):                            # full-batch gradient descent
    logits = model(X)                              # 1. forward pass
    loss = loss_fn(logits, y)                      # 2. compute the loss
    optimizer.zero_grad()                          # 3. clear old gradients                     -> Ch. 2
    loss.backward()                                # 4. backpropagation: compute all gradients -> Ch. 2
    optimizer.step()                               # 5. update every parameter
    if step % 100 == 0:
        print(f"step {step:3d}  loss {loss.item():.4f}")

with torch.no_grad():                              # no gradients needed for evaluation        -> Ch. 2
    accuracy = (model(X).argmax(dim=1) == y).float().mean()
print(f"training accuracy: {accuracy.item():.1%}")

In [ ]:
# Visualise what the network learned: colour every point of a grid by the predicted class.
xx, yy = torch.meshgrid(torch.linspace(-1.5, 2.5, 200), torch.linspace(-1.0, 1.5, 200), indexing="xy")
grid = torch.stack([xx.flatten(), yy.flatten()], dim=1)           # (40000, 2)
with torch.no_grad():
    pred = model(grid).argmax(dim=1).reshape(xx.shape)

plt.figure(figsize=(6, 3.5))
plt.contourf(xx, yy, pred, alpha=0.3, cmap="coolwarm")
plt.scatter(X[:, 0], X[:, 1], c=y, s=6, cmap="coolwarm")
plt.title("Decision regions learned by the MLP"); plt.show()

Those five lines inside the loop (**forward, loss, zero_grad, backward, step**) are the heartbeat of almost every PyTorch program you will ever read. Large language models, image generators and the networks in the papers you will implement all train with this same loop. What changes is the model, the data and the bookkeeping around the loop.

The rest of this notebook explains each piece, in the order you need it:

| Line | Concept | Chapter |
| --- | --- | --- |
| `X`, `y`, `grid` | Tensors: arrays with shapes, dtypes and devices | 1 |
| `loss.backward()`, `zero_grad()`, `no_grad()` | Autograd | 2 |
| `optimizer.step()` | What an optimizer replaces | 3 |
| `nn.Sequential`, `nn.Linear` | Modules | 4 |
| `nn.CrossEntropyLoss`, `argmax` | Losses, logits and predictions | 5 |
| *(missing here)* mini-batches, train/validation split | Data pipelines | 6 |
| *(missing here)* honest evaluation on unseen data | A real project | 7 |

Notice that the script above has a flaw that you would never accept in a real project: it reports accuracy **on the training data**. Chapter 7 does this properly.

---
# Chapter 1. Tensors

🎯 **In this chapter you will learn to:** create tensors with the right dtype; read and change shapes; index, broadcast and reduce along a chosen dimension; multiply matrices; and move work to the GPU.

A **tensor** is an n-dimensional array of numbers that all have the same type. If you have used NumPy, you already know most of the API: the names, slicing and broadcasting rules are nearly identical. A PyTorch tensor adds three things a NumPy array lacks:

1. **A device.** A tensor can live in GPU memory, and then every operation on it runs on the GPU.
2. **Automatic differentiation.** A tensor can record the operations applied to it, so that gradients are computed for you (Chapter 2).
3. **Deep-learning number formats**, such as 16-bit floats, which make training faster (Notebook 2).

### Vocabulary: rank and shape
The **rank** (also: number of dimensions, `ndim`) is how many indices you need to pick out one number. The **shape** lists the size along each dimension.

| Rank | Name | Example shape | Typical meaning in deep learning |
| --- | --- | --- | --- |
| 0 | scalar | `()` | a loss value |
| 1 | vector | `(10,)` | a bias vector, or the labels of a batch of 10 examples |
| 2 | matrix | `(64, 784)` | a batch of 64 flattened 28×28 images, **one row per example** |
| 4 | 4-D tensor | `(64, 3, 32, 32)` | a batch of 64 colour images: **(N, C, H, W)** = batch, channels, height, width |

**Convention:** the first dimension is almost always the **batch** dimension (`N`). Models process many examples at once, and every example sits on its own "row" along dimension 0.

Most bugs in deep-learning code are shape bugs. That is why this chapter is long, and why the habit of **printing shapes** is the most valuable one you will build.

## 1.1 Creating tensors

In [ ]:
a = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])          # from a nested Python list
print(a)
print("shape:", a.shape, "| ndim:", a.ndim, "| numel:", a.numel())
print("dtype:", a.dtype, "| device:", a.device)

`a.shape` is a `torch.Size`, which behaves like a tuple: `a.shape[0]` is 2 and `a.shape[1]` is 3. `a.numel()` is the total number of elements (2 × 3 = 6).

The **factory functions** create tensors of a given shape. These are the ones you will meet constantly:

In [ ]:
print(torch.zeros(2, 3))                    # all zeros, shape (2, 3)
print(torch.ones(2, 3))
print(torch.full((2, 2), 7.0))              # every entry 7.0
print(torch.arange(0, 10, 2))               # 0, 2, 4, 6, 8 : like range(), the end is excluded
print(torch.linspace(0, 1, 5))              # 5 evenly spaced points, both ends included
print(torch.eye(3))                         # 3x3 identity matrix
print(torch.rand(2, 3))                     # uniform random numbers in [0, 1)
print(torch.randn(2, 3))                    # standard normal N(0, 1): used to initialise weights
print(torch.randint(0, 10, (5,)))           # 5 random integers in [0, 10): fake class labels
print(torch.zeros_like(a))                  # same shape, dtype and device as `a`

### dtypes: the first place PyTorch differs from NumPy
Every tensor has one **dtype** (data type). The four you need now:

| dtype | Python alias | Used for |
| --- | --- | --- |
| `torch.float32` | `torch.float` | **Everything continuous:** inputs, weights, activations, losses |
| `torch.int64` | `torch.long` | **Class labels** and indices |
| `torch.bool` | | Masks, such as `x > 0` |
| `torch.float64` | `torch.double` | Rarely: high-precision checks. NumPy's default! |

PyTorch creates floating-point tensors as `float32` by default, because GPUs are fast at 32-bit arithmetic and neural networks do not need 64-bit precision. NumPy defaults to `float64`. When NumPy data meets a PyTorch model, you get one of the most common first-week errors:

In [ ]:
x_np = np.random.rand(4, 3)                  # NumPy: float64
layer = nn.Linear(3, 2)                      # PyTorch layer: float32 weights
print("numpy dtype:", x_np.dtype, "| layer dtype:", layer.weight.dtype)

try:
    layer(torch.from_numpy(x_np))            # float64 input into a float32 layer
except RuntimeError as e:
    print("RuntimeError:", e)

x = torch.from_numpy(x_np).float()           # the fix: convert ONCE, where NumPy data enters PyTorch
print("works:", layer(x).shape)

Converting between dtypes: `x.float()`, `x.long()`, `x.bool()`, or in general `x.to(torch.float32)`.

⚠️ **Pitfall: integer division.** `torch.tensor(7) / 2` gives `3.5` (true division always returns a float), but `torch.tensor(7) // 2` gives `3`. Accuracy computed as `correct.sum() / n` is therefore fine.

### NumPy and PyTorch: copy or share?
- `torch.tensor(array)` **always copies** the data.
- `torch.from_numpy(array)` and `tensor.numpy()` **share memory** (on the CPU). Changing one changes the other. Sharing is fast, but it can surprise you:

In [ ]:
arr = np.zeros(3)
shared = torch.from_numpy(arr)
copied = torch.tensor(arr)
arr[0] = 42.0                    # modify the NumPy array
print("shared:", shared)         # sees the change
print("copied:", copied)         # does not

## 1.2 Changing shape: `reshape`, `flatten`, `unsqueeze`, `squeeze`, `permute`

Reshaping never changes the numbers or their count, only how they are arranged into dimensions. Imagine the elements laid out in one long row, in **row-major order** (the last index changes fastest). `reshape` cuts that row into a new grid.

| Operation | What it does | Typical use |
| --- | --- | --- |
| `x.reshape(2, 3)` | Same elements, new shape. One size may be `-1`, meaning "work it out" | General reshaping |
| `x.flatten(start_dim=1)` | Merge every dimension from `start_dim` on into one | `(N, 1, 28, 28) → (N, 784)` before an MLP |
| `x.unsqueeze(d)` | Insert a new dimension of size 1 at position `d` | Turn one example into a batch of one |
| `x.squeeze(d)` | Remove dimension `d` if its size is 1 | `(N, 1) → (N,)` |
| `x.permute(…)` | Reorder the dimensions | `(H, W, C) → (C, H, W)` for images |
| `x.T` / `x.transpose(i, j)` | Swap two dimensions | Matrix transpose |

🤔 **Predict** the shapes printed by the next cell before you run it.

In [ ]:
x = torch.arange(12)                  # tensor([0, 1, ..., 11]), shape (12,)
print(x.reshape(3, 4))                # rows filled in order: [0..3], [4..7], [8..11]
print(x.reshape(2, -1).shape)         # -1 is inferred: 12 / 2 = 6
print(x.reshape(2, 3, 2).shape)

imgs = torch.randn(8, 1, 28, 28)      # a batch of 8 grayscale images, (N, C, H, W)
print("flatten(start_dim=1):", imgs.flatten(start_dim=1).shape)   # keep N, merge the rest
print("flatten():           ", imgs.flatten().shape)              # merges the batch too: usually a bug!

In [ ]:
one_image = torch.randn(1, 28, 28)                 # a single image, (C, H, W)
batch = one_image.unsqueeze(0)                     # models always expect a batch dimension
print("unsqueeze(0):", one_image.shape, "->", batch.shape)
print("same thing with None indexing:", one_image[None].shape)

preds = torch.randn(5, 1)                          # e.g. a regression model's output
print("squeeze(1):", preds.shape, "->", preds.squeeze(1).shape)

photo = torch.rand(32, 32, 3)                      # how image files are stored: (H, W, C)
chw = photo.permute(2, 0, 1)                       # new order: old dim 2, then old 0, then old 1
print("permute(2, 0, 1):", photo.shape, "->", chw.shape)

⚠️ **Pitfall: `reshape` is not `permute`.** Both can turn shape `(32, 32, 3)` into `(3, 32, 32)`, but only `permute` moves each pixel's three colour values into the three channels. `reshape` just re-cuts the long row of numbers, which scrambles the image. The cell below shows the difference on a tiny "image" whose values reveal where they came from.

In [ ]:
# A 2x2 "image" with 3 channels, stored (H, W, C). The value encodes the channel: 0.x = R, 1.x = G, 2.x = B
img = torch.tensor([[[0.0, 1.0, 2.0], [0.1, 1.1, 2.1]],
                    [[0.2, 1.2, 2.2], [0.3, 1.3, 2.3]]])
print("permute -> channel 0 should hold only 0.x values:\n", img.permute(2, 0, 1)[0])
print("reshape -> channel 0 is a mixture (wrong):\n", img.reshape(3, 2, 2)[0])

You will also meet `x.view(...)` in other people's code. It does the same as `reshape` in most cases, but refuses to work on some tensors (for example, right after a transpose). Notebook 2 explains why, using **strides**. Until then, prefer `reshape`.

## 1.3 Indexing and slicing
Indexing works like NumPy: one index (or slice) per dimension, separated by commas.

In [ ]:
x = torch.arange(12).reshape(3, 4)
print(x)
print("x[1]        ->", x[1])              # row 1 (the second row)
print("x[1, 2]     ->", x[1, 2])           # a 0-d tensor; use .item() to get a Python number
print("x[:, 2]     ->", x[:, 2])           # column 2, from every row
print("x[:, -1]    ->", x[:, -1])          # the last column
print("x[0:2, 1:3] ->\n", x[0:2, 1:3])     # rows 0-1, columns 1-2
print("x[[0, 2]]   ->\n", x[[0, 2]])       # a list of indices picks those rows

**Boolean masks** select every element where a condition is true. They are how you count correct predictions, filter examples by label, or clip values.

In [ ]:
x = torch.arange(12).reshape(3, 4)
mask = x > 5
print(mask)
print("x[mask]:", x[mask])              # a 1-D tensor of the selected values
print("how many > 5:", mask.sum().item())

y = x.clone()                           # clone() makes an independent copy
y[y % 2 == 0] = -1                      # assign through a mask
print(y)

### A pattern you will use constantly: one entry per row
Suppose a classifier produced probabilities for 4 examples and 3 classes, and you want the probability it gave to the **correct** class of each example. You need entry `(0, label[0])`, entry `(1, label[1])`, and so on. Index with two tensors of equal length: one for the rows, one for the columns.

In [ ]:
probs = torch.tensor([[0.7, 0.2, 0.1],
                      [0.1, 0.8, 0.1],
                      [0.3, 0.3, 0.4],
                      [0.5, 0.4, 0.1]])
labels = torch.tensor([0, 1, 0, 1])                 # correct class of each example

p_correct = probs[torch.arange(4), labels]          # rows 0,1,2,3 paired with columns 0,1,0,1
print("probability of the correct class:", p_correct)   # tensor([0.7, 0.8, 0.3, 0.4])
print("cross-entropy by hand:", -torch.log(p_correct).mean())

You have just computed cross-entropy by hand. In other people's code you will often see the same selection written as `probs.gather(1, labels.unsqueeze(1))`, which does the same thing.

## 1.4 Broadcasting
When you combine two tensors of **different shapes** with an elementwise operation (`+ - * /`, comparisons, …), PyTorch tries to *broadcast* them: it stretches size-1 dimensions, without copying any data, so that the shapes match.

**The rule.** Line the two shapes up **from the right**. Compare each pair of sizes:
- equal sizes are fine;
- if one size is 1, it is stretched to match the other;
- a missing dimension (on the left) counts as size 1;
- anything else is an error.

| Shape A | Shape B | Aligned from the right | Result |
| --- | --- | --- | --- |
| `(64, 10)` | `(10,)` | `(64, 10)` vs `( 1, 10)` | `(64, 10)`: B is added to every row |
| `(64, 10)` | `(64, 1)` | `(64, 10)` vs `(64,  1)` | `(64, 10)`: B is added to every column |
| `(3, 1)` | `(1, 4)` | both stretched | `(3, 4)`: every pair of values |
| `(64, 10)` | `(64,)` | `(64, 10)` vs `( 1, 64)` | **error**: 10 ≠ 64 |

You use broadcasting every time a layer adds its bias: `Z = X @ W.T + b` adds `b`, of shape `(out,)`, to every row of `X @ W.T`, of shape `(N, out)`.

In [ ]:
X = torch.randn(5, 3)                                 # 5 examples, 3 features
b = torch.tensor([10.0, 20.0, 30.0])                  # one value per feature
print((X + b).shape)                                  # (5, 3): b added to every row

# Standardise every feature (column) to mean 0, std 1.
mean = X.mean(dim=0, keepdim=True)                    # (1, 3): one mean per column (see 1.5)
std = X.std(dim=0, keepdim=True)                      # (1, 3)
Xs = (X - mean) / std                                 # (5, 3) op (1, 3) -> (5, 3)
print("column means:", Xs.mean(dim=0))                # approx. 0
print("column stds: ", Xs.std(dim=0))                 # approx. 1

# The "outer" pattern: (3, 1) op (1, 4) -> (3, 4), every combination
rows = torch.tensor([[1.0], [2.0], [3.0]])
cols = torch.tensor([[10.0, 20.0, 30.0, 40.0]])
print(rows * cols)

⚠️ **Pitfall: the `(N, 1)` versus `(N,)` bug.** This is the single most common silent bug in regression code. A regression model outputs shape `(N, 1)`, with one column. Targets loaded from a file often have shape `(N,)`. Now apply the rule: `(N, 1)` minus `(N,)` aligns as `(N, 1)` vs `(1, N)`, so **both** are stretched and the result is `(N, N)`. Every prediction is compared with every target. The loss is a finite number, training runs, and the model learns almost nothing.

🤔 **Predict** the shape of `pred - y` before running the cell.

In [ ]:
N = 4
pred = torch.randn(N, 1)       # model output: (N, 1)
y = torch.randn(N)             # targets:      (N,)
print("pred - y has shape", (pred - y).shape)            # (4, 4), not (4,)!

wrong = ((pred - y) ** 2).mean()                         # silently wrong
right = ((pred.squeeze(1) - y) ** 2).mean()              # make the shapes equal first
print(f"wrong loss: {wrong:.3f}   right loss: {right:.3f}")

The defence is a habit: **assert shapes where data meets the model or the loss**, for example `assert pred.shape == y.shape`. It costs one line and catches this whole class of bugs.

## 1.5 Reductions and the meaning of `dim`
A **reduction** collapses a dimension by combining its values: `sum`, `mean`, `max`, `min`, `std`, `argmax`. The argument `dim` says **which dimension disappears**.

For a matrix of shape `(examples, features)`:
- `x.sum(dim=0)` collapses the examples, giving **one value per feature** (a column total), shape `(features,)`;
- `x.sum(dim=1)` collapses the features, giving **one value per example** (a row total), shape `(examples,)`.

A reliable way to remember it: *the dimension you name is the one that goes away*. With `keepdim=True` it stays, with size 1, which keeps the result broadcastable against the original (as in the standardisation example above).

In [ ]:
x = torch.tensor([[1.0, 5.0, 3.0],
                  [4.0, 2.0, 6.0]])               # shape (2, 3)
print("x.sum()              =", x.sum())               # everything -> a scalar
print("x.sum(dim=0)         =", x.sum(dim=0))          # (3,): one per column
print("x.sum(dim=1)         =", x.sum(dim=1))          # (2,): one per row
print("x.sum(dim=1, keepdim=True).shape =", x.sum(dim=1, keepdim=True).shape)   # (2, 1)

values, indices = x.max(dim=1)                     # max along a dim returns BOTH
print("max of each row:", values, "found at column", indices)
print("argmax(dim=1):  ", x.argmax(dim=1))         # how logits become predicted classes

Here is the pattern that turns a classifier's output into an accuracy. You will write it in every classification project.

In [ ]:
logits = torch.tensor([[2.0, 0.5, -1.0],     # model scores, shape (N=4, C=3)
                       [0.1, 0.2,  3.0],
                       [1.0, 2.0,  0.0],
                       [0.3, 0.1,  0.2]])
labels = torch.tensor([0, 2, 0, 0])

predicted = logits.argmax(dim=1)              # (4,): the class with the highest score in each row
correct = (predicted == labels)               # (4,) booleans
print("predicted:", predicted, "| correct:", correct)
print("accuracy:", correct.float().mean().item())   # bool -> float, then average

## 1.6 Combining and selecting: `cat`, `stack`, `where`, `masked_fill`, `topk`, `sort`
These six functions appear in almost every code base. The first two **combine** tensors; the other four **select** values.

| Function | What it does | Shapes |
| --- | --- | --- |
| `torch.cat([a, b], dim=d)` | Joins tensors **along an existing** dimension | `(2, 3)` and `(4, 3)` with `dim=0` → `(6, 3)` |
| `torch.stack([a, b], dim=d)` | Joins equal-shape tensors along a **new** dimension | two `(2, 3)` with `dim=0` → `(2, 2, 3)` |
| `torch.where(cond, a, b)` | Elementwise: `a` where `cond` is true, else `b` | broadcast of the three |
| `x.masked_fill(mask, value)` | Copy of `x` with `value` wherever `mask` is true | same as `x` |
| `x.topk(k, dim)` | The `k` largest values and their indices | `k` along `dim` |
| `x.sort(dim)` / `x.argsort(dim)` | Sorted values and indices / only the indices | same as `x` |

🤔 **Predict** the four shapes printed by the first two lines of the next cell.

In [ ]:
a, b = torch.zeros(2, 3), torch.ones(4, 3)
print("cat dim=0:  ", torch.cat([a, b], dim=0).shape)                 # (6, 3): more rows
print("cat dim=1:  ", torch.cat([a, torch.ones(2, 5)], dim=1).shape)  # (2, 8): more columns
p, q = torch.zeros(2, 3), torch.ones(2, 3)
print("stack dim=0:", torch.stack([p, q], dim=0).shape)              # (2, 2, 3): a new leading dimension
print("stack dim=2:", torch.stack([p, q], dim=2).shape)              # (2, 3, 2): a new last dimension

# stack is what a DataLoader does to build a batch: 4 examples of shape (1, 28, 28) -> (4, 1, 28, 28)
examples = [torch.randn(1, 28, 28) for _ in range(4)]
print("a batch:    ", torch.stack(examples).shape)

In [ ]:
x = torch.tensor([[-2.0, 0.5, 3.0],
                  [1.0, -0.5, -4.0]])
print("where(x > 0, x, 0):\n", torch.where(x > 0, x, torch.tensor(0.0)))      # ReLU written by hand
print("masked_fill(x < 0, -1):\n", x.masked_fill(x < 0, -1.0))

# A very common pattern: forbid some options by setting their logits to -inf before a softmax
logits = torch.tensor([[2.0, 1.0, 0.5, 3.0]])
allowed = torch.tensor([[True, True, True, False]])                  # option 3 is not allowed here
print("softmax with option 3 masked out:", torch.softmax(logits.masked_fill(~allowed, float("-inf")), dim=1))

In [ ]:
scores = torch.tensor([[0.1, 0.5, 0.3, 0.9, 0.2],
                       [0.8, 0.1, 0.6, 0.0, 0.4]])
values, indices = scores.topk(3, dim=1)            # the 3 best classes of each row, best first
print("top-3 values: ", values)
print("top-3 classes:", indices)
print("full ranking: ", scores.argsort(dim=1, descending=True))

labels = torch.tensor([1, 4])
in_top3 = (indices == labels[:, None]).any(dim=1)  # broadcasting (N, 3) == (N, 1)
print("top-3 accuracy:", in_top3.float().mean().item())   # common on ImageNet-style benchmarks

## 1.7 Elementwise versus matrix multiplication
- `A * B` multiplies **elementwise** (with broadcasting). In math this is the Hadamard product, `A ⊙ B`.
- `A @ B` is **matrix multiplication**: `(n, k) @ (k, m) → (n, m)`. The inner sizes must match.
- With more than two dimensions, `@` performs a **batched** matrix multiplication: the last two dimensions are multiplied as matrices, and the leading dimensions act as a batch (and broadcast).

This lets us confirm exactly what a fully connected layer computes. `nn.Linear(in_features, out_features)` stores a weight matrix of shape **`(out, in)`** and computes `y = x @ W.T + b`:

In [ ]:
lin = nn.Linear(3, 2)                       # 3 inputs -> 2 outputs
x = torch.randn(5, 3)                       # a batch of 5 examples
print("weight:", lin.weight.shape, "| bias:", lin.bias.shape)    # (2, 3) and (2,)

by_hand = x @ lin.weight.T + lin.bias       # (5,3) @ (3,2) -> (5,2), plus (2,) broadcast to every row
print("output:", lin(x).shape, "| matches by hand:", torch.allclose(lin(x), by_hand))

A = torch.randn(10, 4, 5)                   # a batch of 10 matrices, each 4x5
B = torch.randn(10, 5, 6)
print("batched matmul:", (A @ B).shape)     # (10, 4, 6)

`torch.allclose(a, b)` checks that two tensors are equal **up to floating-point rounding**. Never compare floats with `==`, because two mathematically equal computations done in a different order differ in the last few bits.

You will also see `torch.einsum("ij,jk->ik", A, B)` in papers' code. It is a compact language for products and sums like these, and Notebook 2 teaches it properly.

## 1.8 Devices: CPU and GPU
Every tensor lives on exactly one device. Three rules cover almost everything:

1. **Move with `.to(device)`.** It returns a copy on the target device (or the tensor itself, if it is already there).
2. **All tensors in one operation must be on the same device.** Otherwise you get the most common error of your first month: *"Expected all tensors to be on the same device"*.
3. **Copying between CPU and GPU is slow** compared with computing. Move each batch to the GPU once, keep the work there, and bring back only what you need: `.item()` for a single number, `.cpu()` for a tensor.

In [ ]:
x_cpu = torch.randn(3, 3)
x_dev = x_cpu.to(device)
print("x_dev lives on:", x_dev.device)

if device.type != "cpu":
    try:
        x_cpu + x_dev                               # mixed devices
    except RuntimeError as e:
        print("RuntimeError:", str(e)[:95], "...")

y = torch.zeros(3, 3, device=device)                # or create a tensor directly on the device
loss = ((x_dev - y) ** 2).mean()
print("a Python float:", loss.item())               # 1-element tensor -> Python number
print("a NumPy array:\n", x_dev.cpu().numpy())      # GPU tensors must come to the CPU before .numpy()

Why bother with a GPU? A GPU has thousands of simple cores that can do the same operation on many numbers in parallel, and neural networks are mostly large matrix multiplications. The cell below compares one large matrix multiplication on both devices.

🏭 **Timing GPU code correctly.** GPU work runs **asynchronously**. Python queues the work and immediately moves on, so a naive timer measures how long it took to *queue* the work, not to *do* it. `torch.cuda.synchronize()` waits until the GPU has finished. The first GPU call also pays one-off start-up costs, so do a warm-up run before timing.

In [ ]:
def time_matmul(dev, n=2048, reps=10):
    a, b = torch.randn(n, n, device=dev), torch.randn(n, n, device=dev)
    a @ b                                            # warm-up
    if dev.type == "cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(reps):
        a @ b
    if dev.type == "cuda": torch.cuda.synchronize()  # wait for the GPU to actually finish
    return (time.perf_counter() - t0) / reps

t_cpu = time_matmul(torch.device("cpu"))
print(f"CPU: {t_cpu * 1e3:8.2f} ms per 2048x2048 matrix multiplication")
if device.type == "cuda":
    t_gpu = time_matmul(device)
    print(f"GPU: {t_gpu * 1e3:8.2f} ms  (about {t_cpu / t_gpu:.0f}x faster)")

## 1.9 In-place operations
Methods whose name ends in an underscore modify the tensor **in place**: `x.add_(1)`, `x.mul_(2)`, `x.zero_()`. So do `x += 1` and `x[idx] = value`. Everything else returns a new tensor and leaves the input alone.

In-place operations save memory, but they can destroy values that autograd needs for the backward pass (you will see the error in Chapter 2). **Default to the out-of-place versions.** You will mainly see the underscore versions inside library code, such as initialisation functions (`nn.init.zeros_`).

In [ ]:
x = torch.ones(3)
y = x.add(1)        # new tensor; x unchanged
print("after add: ", x, y)
x.add_(1)           # x itself is modified
print("after add_:", x)

## 📖 Code-reading drill
Here is a line in the style you will meet in many repositories. Work out the answer on paper, then check it.

```python
feats = torch.randn(32, 3, 8, 8)
out = feats.flatten(2).mean(-1)
```

**(a)** What is the shape of `feats.flatten(2)`? **(b)** What is the shape of `out`, and what does each of its numbers mean?

In [ ]:
feats = torch.randn(32, 3, 8, 8)
print(feats.flatten(2).shape, feats.flatten(2).mean(-1).shape)
# (a) (32, 3, 64): every dimension from 2 on (8x8) is merged.
# (b) (32, 3): the average value of each channel of each image.
#     dim=-1 means "the last dimension"; negative dims count from the end.
#     This is "global average pooling", which you will meet again with CNNs in Notebook 2.

## 🧪 Exercises

**Exercise 1.1: pairwise squared distances with broadcasting.** `★★☆ · ~15 min`
Write `pairwise_sq_dists(A, B)` that takes `A` of shape `(n, d)` and `B` of shape `(m, d)` and returns `D` of shape `(n, m)` with `D[i, j] = ‖A[i] − B[j]‖²`. Use broadcasting, not Python loops. (This is the core of k-nearest-neighbours and k-means.)

In [ ]:
# ✍️ Your attempt here.
# Hint: A[:, None, :] has shape (n, 1, d) and B[None, :, :] has shape (1, m, d). Subtract, square, then sum over the last dim.

In [ ]:
# 🔎 Self-check for Exercise 1.1 (tests YOUR function)
def _test(fn):
    A, B = torch.randn(5, 3), torch.randn(7, 3)
    D = fn(A, B)
    assert D.shape == (5, 7), f"shape {tuple(D.shape)}, expected (5, 7)"
    loop = torch.tensor([[((a - b) ** 2).sum().item() for b in B] for a in A])
    assert torch.allclose(D, loop, atol=1e-5), "values differ from a Python-loop reference"
self_check("pairwise_sq_dists", _test)

In [ ]:
# @title ✅ Solution (double-click to reveal) { display-mode: "form" }
# Solutions only execute when RUN_SOLUTIONS = True (setup cell), so "Run all" stays fast.
if RUN_SOLUTIONS:
    def pairwise_sq_dists(A, B):
        diff = A[:, None, :] - B[None, :, :]     # (n, 1, d) - (1, m, d) -> (n, m, d)
        return (diff ** 2).sum(dim=-1)           # sum over d -> (n, m)

    A, B = torch.randn(5, 3), torch.randn(7, 3)
    print(pairwise_sq_dists(A, B).shape)
    # Memory note: the intermediate (n, m, d) tensor can be huge for big n, m.
    # The identity ||a-b||² = ||a||² + ||b||² - 2 a·b avoids it, and torch.cdist(A, B) ** 2 does it for you.
    print(torch.allclose(pairwise_sq_dists(A, B), torch.cdist(A, B) ** 2, atol=1e-5))
else:
    print("Solution not executed (RUN_SOLUTIONS = False).")

**Exercise 1.2: a numerically stable softmax.** `★★☆ · ~15 min`
The softmax of a row `z` is `exp(z_k) / Σ_j exp(z_j)`. Computed literally, `exp(1000)` overflows to `inf` and the result becomes `nan`. Write `softmax_stable(z, dim)` that works for any `z` along any `dim`.
Use the fact that `softmax(z) = softmax(z − c)` for any constant `c`. Subtracting the row maximum makes the largest exponent `exp(0) = 1`.

In [ ]:
# ✍️ Your attempt here.
# Hint: z.max(dim=dim, keepdim=True).values gives the maximum with the dimension kept, so it broadcasts.

In [ ]:
# 🔎 Self-check for Exercise 1.2
def _test(fn):
    z = torch.tensor([[1000.0, 1001.0, 1002.0], [-5.0, 0.0, 5.0]])
    s = fn(z, dim=1)
    assert torch.isfinite(s).all(), "result contains inf or nan"
    assert torch.allclose(s, torch.softmax(z, dim=1), atol=1e-6), "values differ from torch.softmax"
    z2 = torch.randn(4, 6)
    assert torch.allclose(fn(z2, dim=0), torch.softmax(z2, dim=0), atol=1e-6), "wrong along dim=0"
self_check("softmax_stable", _test)

In [ ]:
# @title ✅ Solution (double-click to reveal) { display-mode: "form" }
# Solutions only execute when RUN_SOLUTIONS = True (setup cell), so "Run all" stays fast.
if RUN_SOLUTIONS:
    def softmax_stable(z, dim):
        z = z - z.max(dim=dim, keepdim=True).values      # largest entry becomes 0 -> no overflow
        e = torch.exp(z)
        return e / e.sum(dim=dim, keepdim=True)           # keepdim=True so the division broadcasts

    print(softmax_stable(torch.tensor([[1000.0, 1001.0, 1002.0]]), dim=1))
else:
    print("Solution not executed (RUN_SOLUTIONS = False).")

**Exercise 1.3: preparing a batch of images.** `★★☆ · ~15 min`
Image files load as `uint8` arrays with values 0–255 and layout `(N, H, W, C)`. Models want `float32`, layout `(N, C, H, W)`, and per-channel normalisation. Write `prepare_batch(x, mean, std)` that takes a `uint8` tensor of shape `(N, H, W, 3)` and per-channel lists `mean` and `std` (three numbers each, on the [0, 1] scale), and returns `(x / 255 − mean) / std` with layout `(N, 3, H, W)` and dtype `float32`.

In [ ]:
# ✍️ Your attempt here.
# Hint: permute(0, 3, 1, 2) first, then .float() / 255. Reshape mean and std to (1, 3, 1, 1) so they broadcast.

In [ ]:
# 🔎 Self-check for Exercise 1.3
def _test(fn):
    x = torch.randint(0, 256, (2, 4, 5, 3), dtype=torch.uint8)
    out = fn(x, [0.5, 0.4, 0.3], [0.2, 0.25, 0.3])
    assert out.shape == (2, 3, 4, 5), f"shape {tuple(out.shape)}, expected (2, 3, 4, 5)"
    assert out.dtype == torch.float32, f"dtype {out.dtype}, expected float32"
    ref = (x[1, 2, 3, 1].item() / 255 - 0.4) / 0.25
    assert abs(out[1, 1, 2, 3].item() - ref) < 1e-5, "values look wrong (check the channel order)"
self_check("prepare_batch", _test)

In [ ]:
# @title ✅ Solution (double-click to reveal) { display-mode: "form" }
# Solutions only execute when RUN_SOLUTIONS = True (setup cell), so "Run all" stays fast.
if RUN_SOLUTIONS:
    def prepare_batch(x, mean, std):
        x = x.permute(0, 3, 1, 2).float() / 255.0                  # (N, 3, H, W), values in [0, 1]
        mean = torch.tensor(mean, dtype=torch.float32).reshape(1, 3, 1, 1)   # broadcastable shapes
        std = torch.tensor(std, dtype=torch.float32).reshape(1, 3, 1, 1)
        return (x - mean) / std

    print(prepare_batch(torch.randint(0, 256, (2, 4, 5, 3), dtype=torch.uint8), [0.5] * 3, [0.25] * 3).shape)
else:
    print("Solution not executed (RUN_SOLUTIONS = False).")

### 📌 Key takeaways
- Tensors have a **shape**, a **dtype** and a **device**. Most errors are a mismatch in one of the three. Print them.
- Continuous data is `float32`, class labels are `int64`. Convert NumPy `float64` data with `.float()`.
- `dim=k` in a reduction means "collapse dimension k". Use `keepdim=True` when you will broadcast the result back.
- Broadcasting aligns shapes **from the right**. `(N, 1)` combined with `(N,)` silently becomes `(N, N)`.
- `cat` joins along an existing dimension, `stack` creates a new one; `where`, `masked_fill` and `topk` select values without loops.
- `nn.Linear` computes `x @ W.T + b` with `W` of shape `(out, in)`.
- Keep data on one device; bring back scalars with `.item()`.

### ✔️ Check yourself

Commit to an answer (say it out loud or write it down) *before* you expand it.

**1.** What is the shape of `torch.randn(64, 3, 32, 32).mean(dim=(2, 3))`, and what does it represent?

<details>
<summary>Answer</summary>

`(64, 3)`: the mean of each channel of each image. Dimensions 2 and 3 (height and width) are collapsed.

</details>

**2.** `a` has shape `(8, 1)` and `b` has shape `(8,)`. What is the shape of `a + b`?

<details>
<summary>Answer</summary>

`(8, 8)`. Aligned from the right, `(8, 1)` vs `(1, 8)`: both size-1 dimensions are stretched. This is the classic silent regression bug.

</details>

**3.** Why does `layer(torch.from_numpy(np_array))` often fail, and what is the fix?

<details>
<summary>Answer</summary>

NumPy arrays default to `float64`, while PyTorch layers use `float32`. Convert once at the boundary: `torch.from_numpy(arr).float()`.

</details>

**4.** You have a list of 16 tensors of shape `(3, 32, 32)`. Which call makes a batch of shape `(16, 3, 32, 32)`, `torch.cat` or `torch.stack`?

<details>
<summary>Answer</summary>

`torch.stack(list, dim=0)`, which adds a new dimension. `torch.cat(list, dim=0)` would give `(48, 32, 32)`.

</details>

**5.** You want a `(3, 32, 32)` image tensor from an array of shape `(32, 32, 3)`. Which operation, `reshape` or `permute`?

<details>
<summary>Answer</summary>

`permute(2, 0, 1)`. `reshape` would re-cut the flat sequence of numbers and mix the colour channels.

</details>

**6.** Why should you call `torch.cuda.synchronize()` before stopping a timer around GPU code?

<details>
<summary>Answer</summary>

GPU operations are asynchronous. Without synchronising, you measure the time to queue the work, not the time to do it.

</details>

---
# Chapter 2. Autograd: backpropagation, done for you

🎯 **In this chapter you will learn to:** compute gradients with `backward()`; explain what autograd records and stores; avoid the gradient-accumulation and memory-leak pitfalls; switch autograd off when you do not need it; and run gradient descent with nothing but tensors.

Training a network means computing the gradient of the loss with respect to every parameter, then taking a small step downhill. In your course you computed these gradients by hand with **backpropagation**, which is the chain rule applied backwards through the computation.

**Autograd** is PyTorch's implementation of exactly that algorithm, written once for every operation in the library. While you compute with tensors marked `requires_grad=True`, PyTorch quietly records which operations produced which tensors. Calling `.backward()` on the final scalar walks that record backwards, applying the chain rule, and stores each gradient in the `.grad` attribute of the corresponding tensor.

## 2.1 A hand calculation, checked by autograd

Take the smallest interesting network: one input, one sigmoid hidden unit, one sigmoid output unit, and a squared-error loss.

$$z_1 = w_1 x + b_1,\quad a_1 = \sigma(z_1),\quad z_2 = w_2 a_1 + b_2,\quad a_2 = \sigma(z_2),\quad L = \tfrac12 (a_2 - y)^2$$

with $w_1 = 0.5,\ b_1 = 0.1,\ w_2 = 0.8,\ b_2 = -0.3$, input $x = 2$ and target $y = 1$. Recall that $\sigma'(z) = \sigma(z)\,(1 - \sigma(z))$.

**Forward pass.** $z_1 = 1.1$, $a_1 = \sigma(1.1) = 0.7503$, $z_2 = 0.8 \cdot 0.7503 - 0.3 = 0.3002$, $a_2 = \sigma(0.3002) = 0.5745$, $L = \tfrac12(0.5745 - 1)^2 = 0.0905$.

**Backward pass** (chain rule, from the loss backwards):

| Quantity | Formula | Value |
| --- | --- | --- |
| $\delta_2 = \partial L/\partial z_2$ | $(a_2 - y)\, a_2 (1 - a_2)$ | $-0.4255 \cdot 0.5745 \cdot 0.4255 = -0.1040$ |
| $\partial L/\partial w_2$ | $\delta_2 \, a_1$ | $-0.0780$ |
| $\partial L/\partial b_2$ | $\delta_2$ | $-0.1040$ |
| $\delta_1 = \partial L/\partial z_1$ | $\delta_2 \, w_2 \, a_1 (1 - a_1)$ | $-0.1040 \cdot 0.8 \cdot 0.7503 \cdot 0.2497 = -0.0156$ |
| $\partial L/\partial w_1$ | $\delta_1 \, x$ | $-0.0312$ |
| $\partial L/\partial b_1$ | $\delta_1$ | $-0.0156$ |

Now let autograd do it. Read the code: the forward pass is the same five equations, written with tensors.

In [ ]:
# Parameters: the tensors we want gradients for -> requires_grad=True
w1 = torch.tensor(0.5, requires_grad=True)
b1 = torch.tensor(0.1, requires_grad=True)
w2 = torch.tensor(0.8, requires_grad=True)
b2 = torch.tensor(-0.3, requires_grad=True)
x, y = torch.tensor(2.0), torch.tensor(1.0)        # data: we do not need gradients for these

# Forward pass: autograd records every operation as it happens
z1 = w1 * x + b1
a1 = torch.sigmoid(z1)
z2 = w2 * a1 + b2
a2 = torch.sigmoid(z2)
L = 0.5 * (a2 - y) ** 2
print(f"forward: z1={z1.item():.4f}  a1={a1.item():.4f}  z2={z2.item():.4f}  a2={a2.item():.4f}  L={L.item():.4f}")

L.backward()          # backward pass: fills .grad of every tensor created with requires_grad=True

for name, p in [("w2", w2), ("b2", b2), ("w1", w1), ("b1", b1)]:
    print(f"dL/d{name} = {p.grad.item():+.4f}")

The numbers match the table to every printed digit. From now on you will rarely compute a gradient by hand. But you must always know **what** autograd is computing, because that is how you debug it when training goes wrong.

## 2.2 What autograd records: the computational graph

- A **leaf** tensor is one you created directly, rather than as the result of an operation. Parameters are leaves.
- Every tensor produced by an operation on a tensor that requires gradients gets a **`grad_fn`**: a small object that knows the local derivative of that operation and points back to the `grad_fn`s of its inputs. Together these form the **computational graph**.
- `backward()` starts at the loss and walks the graph in reverse. At each node it multiplies the incoming gradient by the local derivative and passes the result on, exactly as in the table above.
- Gradients are **stored only for leaves** with `requires_grad=True`. Intermediate gradients (such as $\delta_1$) are computed, used and thrown away to save memory.

In [ ]:
print("L.grad_fn        :", L.grad_fn)                   # the last operation: multiplication by 0.5
print("its inputs       :", L.grad_fn.next_functions)    # links back towards the start of the graph
print("w1 is a leaf?    :", w1.is_leaf, "| a1 is a leaf?", a1.is_leaf)
print("a1.grad          :", a1.grad)                     # None: intermediate gradients are not kept
print("x.requires_grad  :", x.requires_grad)             # data does not need gradients

(Reading `a1.grad` prints a warning that explains the same thing. If you ever need an intermediate gradient, call `a1.retain_grad()` before `backward()`.)

**The graph is rebuilt on every forward pass.** PyTorch records whatever operations *actually ran*. This is called a **dynamic graph** (or "define-by-run"), and it means ordinary Python `if` statements and loops work inside models. The gradient follows whichever branch was taken:

In [ ]:
def f(x):
    if x > 0:                      # plain Python control flow
        return x ** 2              # derivative 2x
    return -3 * x                  # derivative -3

for value in (2.0, -1.0):
    x = torch.tensor(value, requires_grad=True)
    f(x).backward()
    print(f"x = {value:+.0f}: gradient = {x.grad.item()}")

## 2.3 Gradients accumulate, which is why `zero_grad()` exists

🤔 **Predict** what the next cell prints. The derivative of $w^2$ at $w = 3$ is $6$, and we call `backward()` three times.

In [ ]:
w = torch.tensor(3.0, requires_grad=True)
for step in range(3):
    loss = w ** 2
    loss.backward()
    print(f"after backward #{step + 1}: w.grad = {w.grad.item()}")

**`backward()` adds to `.grad`; it never overwrites it.** That is a deliberate design choice. It lets you add up gradients from several small batches before taking one step, which is useful when a large batch does not fit in GPU memory (Notebook 2 shows this "gradient accumulation" trick).

The price is that **you must clear the gradients before every new step**. Forgetting to do so is one of the classic beginner bugs: each step then uses the sum of all past gradients, and training goes haywire. Clearing is what `optimizer.zero_grad()` does. By hand, you set `.grad` to `None`:

In [ ]:
w.grad = None              # what optimizer.zero_grad() does for every parameter
(w ** 2).backward()
print("after clearing:", w.grad.item())

## 2.4 Two rules of `backward()`

**Rule 1: `backward()` needs a scalar.** A gradient $\partial L / \partial w$ is defined for a single number $L$. A training loss is always one number (the mean over the batch), so this is natural. If you try it on a vector, PyTorch complains.

**Rule 2: a graph can be used for backward only once.** To save memory, the intermediate values stored for the backward pass are freed as soon as `backward()` finishes. In a training loop this is never a problem, because every step runs a fresh forward pass and builds a new graph.

In [ ]:
v = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
out = v ** 2                                   # a vector, not a scalar
try:
    out.backward()
except RuntimeError as e:
    print("Rule 1 -> RuntimeError:", e)
out.sum().backward()                           # reduce to a scalar first
print("d(sum of v²)/dv =", v.grad)             # 2v

w = torch.tensor(3.0, requires_grad=True)
loss = w ** 2
loss.backward()
try:
    loss.backward()                            # second backward through the same graph
except RuntimeError as e:
    print("Rule 2 -> RuntimeError:", str(e)[:100], "...")

## 2.5 Switching autograd off

Recording a graph costs memory and time. Switch it off whenever you do not need gradients: during evaluation, during inference, and when updating parameters by hand.

| Tool | What it does | Use it for |
| --- | --- | --- |
| `with torch.no_grad():` | Nothing computed inside the block is recorded | Evaluation loops, manual parameter updates |
| `with torch.inference_mode():` | Like `no_grad`, slightly faster; the results can never be used with autograd later | Pure prediction |
| `t.detach()` | Returns a tensor with the same data, cut out of the graph | Logging, plotting, or deliberately stopping a gradient |

In [ ]:
w = torch.randn(3, requires_grad=True)
y1 = w * 2
with torch.no_grad():
    y2 = w * 2
print("normal       -> requires_grad:", y1.requires_grad)   # True: recorded
print("in no_grad   -> requires_grad:", y2.requires_grad)   # False: not recorded
print("detach()     -> requires_grad:", y1.detach().requires_grad)

try:
    y1.numpy()                       # NumPy knows nothing about graphs
except RuntimeError as e:
    print("RuntimeError:", str(e)[:70], "...")
print("fix:", y1.detach().numpy())

⚠️ **Pitfall: the logging memory leak.** A common way to run out of memory is to accumulate the loss *tensor* for logging:

```python
total_loss += loss          # BUG: keeps every step's graph alive -> memory grows every iteration
total_loss += loss.item()   # correct: a plain Python float, no graph attached
```

Because `total_loss` is a tensor connected to every past graph, none of them can be freed. Always log with `.item()`.

## 2.6 Gradient descent with nothing but tensors

You now have everything needed to train a model without any other PyTorch machinery. Minimise

$$f(w) = (w_0 - 3)^2 + 10\,(w_1 + 1)^2,$$

whose minimum is at $w = (3, -1)$. The update rule is $w \leftarrow w - \eta \nabla f(w)$. Two details are essential, and the comments point them out:

In [ ]:
def gradient_descent(lr, steps=100):
    w = torch.zeros(2, requires_grad=True)
    history = []
    for _ in range(steps):
        f = (w[0] - 3) ** 2 + 10 * (w[1] + 1) ** 2
        f.backward()                       # compute the gradient into w.grad
        with torch.no_grad():              # (1) the update itself must NOT be recorded in the graph
            w -= lr * w.grad
        w.grad = None                      # (2) clear the gradient, or the next step adds to it
        history.append(f.item())
    return w.detach(), history

w_final, hist = gradient_descent(lr=0.05)
print("final w:", w_final, "| final f:", hist[-1])
plt.figure(figsize=(5, 3))
plt.semilogy(hist); plt.xlabel("step"); plt.ylabel("f(w), log scale"); plt.title("Gradient descent, lr = 0.05")
plt.show()

Why `torch.no_grad()` around the update? Without it, `w - lr * w.grad` would become part of the graph, and the next `backward()` would try to differentiate through the update itself. PyTorch actually forbids modifying a leaf that requires gradients outside `no_grad`, which protects you from this mistake.

This loop is all an optimizer does. In Chapter 3 you will replace those two marked lines with `optimizer.step()` and `optimizer.zero_grad()`.

## 2.7 In-place operations and autograd
Some local derivatives need values from the forward pass. For example, the sigmoid's derivative $\sigma(z)(1-\sigma(z))$ needs the sigmoid's own output, so autograd saves it. If you then modify that saved tensor in place, the backward pass would use a wrong value. PyTorch detects this and raises an error instead of silently computing a wrong gradient:

In [ ]:
z = torch.randn(3, requires_grad=True)
s = torch.sigmoid(z)          # autograd saves s, because the backward pass needs it
s.mul_(2)                     # in-place change to a saved tensor
try:
    s.sum().backward()
except RuntimeError as e:
    print("RuntimeError:", str(e)[:120], "...")

s = torch.sigmoid(z) * 2      # out-of-place: a new tensor, the saved one is untouched
s.sum().backward()
print("fine:", z.grad)

## 🧪 Exercises

**Exercise 2.1: verify a famous gradient.** `★★☆ · ~15 min`
For softmax followed by cross-entropy, the gradient with respect to the logits has a beautiful form: for one example with label $y$, $\partial L / \partial z = \mathrm{softmax}(z) - \mathrm{onehot}(y)$. For a batch of $N$ examples with the loss **averaged** over the batch, every row gets an extra factor $1/N$.

Verify this with autograd. Create random `logits` of shape `(8, 5)` with `requires_grad=True` and random integer `labels` in `[0, 5)`. Compute `F.cross_entropy(logits, labels)` (it averages over the batch), call `backward()`, and compare `logits.grad` with the formula using `torch.allclose`.

In [ ]:
# ✍️ Your attempt here.
# Hint: F.one_hot(labels, num_classes=5).float() gives the one-hot matrix. Use logits.detach() inside the formula.

In [ ]:
# @title ✅ Solution (double-click to reveal) { display-mode: "form" }
# Solutions only execute when RUN_SOLUTIONS = True (setup cell), so "Run all" stays fast.
if RUN_SOLUTIONS:
    logits = torch.randn(8, 5, requires_grad=True)
    labels = torch.randint(0, 5, (8,))
    F.cross_entropy(logits, labels).backward()            # mean over the batch by default

    expected = (torch.softmax(logits.detach(), dim=1) - F.one_hot(labels, 5).float()) / 8
    print("autograd matches (softmax - onehot) / N:", torch.allclose(logits.grad, expected, atol=1e-6))
    # This is why cross-entropy trains so well: the gradient is simply "predicted minus true",
    # and it never vanishes while the prediction is wrong.
else:
    print("Solution not executed (RUN_SOLUTIONS = False).")

**Exercise 2.2: how large can the learning rate be?** `★★☆ · ~15 min`
Run `gradient_descent` from §2.6 with `lr` = 0.05, 0.09, 0.1 and 0.11, and print the final `w` for each.
1. Which values converge, and which diverge?
2. Explain the threshold. *Hint:* along one coordinate, $f = c\,(w - w^*)^2$ gives the update $w - w^* \leftarrow (1 - 2c\eta)(w - w^*)$. When does $|1 - 2c\eta| < 1$ hold for both coordinates?

In [ ]:
# ✍️ Your attempt here.

In [ ]:
# @title ✅ Solution (double-click to reveal) { display-mode: "form" }
# Solutions only execute when RUN_SOLUTIONS = True (setup cell), so "Run all" stays fast.
if RUN_SOLUTIONS:
    for lr in (0.05, 0.09, 0.1, 0.11):
        w_final, hist = gradient_descent(lr)
        print(f"lr = {lr:<5} final w = {w_final.tolist()}  final f = {hist[-1]:.3g}")
    # Coordinate 0 has c = 1:  factor |1 - 2*lr|   < 1  <=>  lr < 1
    # Coordinate 1 has c = 10: factor |1 - 20*lr|  < 1  <=>  lr < 0.1
    # The STEEPEST direction sets the limit for all of them: lr < 0.1.
    # At lr = 0.1 the factor is exactly -1, so w1 oscillates between two values forever;
    # at lr = 0.11 it grows by 1.2x per step and diverges.
    # At lr = 0.09 both converge: w1 oscillates around -1 while shrinking (factor -0.8 per step),
    # and w0 approaches 3 smoothly (factor 0.82 per step).
    # This is ill-conditioning: one steep direction caps the step size for every direction.
    # Adaptive optimizers such as Adam (Notebook 2) are one answer to it.
else:
    print("Solution not executed (RUN_SOLUTIONS = False).")

**Exercise 2.3: gradient checking.** `★★☆ · ~20 min`
Before autograd existed, people verified backpropagation code with **finite differences**: $\dfrac{\partial f}{\partial w_i} \approx \dfrac{f(w + \varepsilon e_i) - f(w - \varepsilon e_i)}{2\varepsilon}$, where $e_i$ is the $i$-th unit vector. Write `numerical_grad(f, w, eps=1e-6)` that returns this estimate for every entry of a 1-D tensor `w`. Then compare it with autograd on `f(w) = (w.sin() * w.exp()).sum()`.
Use `dtype=torch.float64`: in `float32`, rounding errors swamp the tiny differences.

In [ ]:
# ✍️ Your attempt here.
# Hint: Loop over i; build w_plus = w.clone(); w_plus[i] += eps (and the same with minus).

In [ ]:
# 🔎 Self-check for Exercise 2.3
def _test(fn):
    w = torch.randn(5, dtype=torch.float64)
    g = fn(lambda v: (v ** 3).sum(), w)
    assert g.shape == w.shape, "the result must have the same shape as w"
    assert torch.allclose(g, 3 * w ** 2, atol=1e-5), "wrong values for f(w) = sum(w^3)"
self_check("numerical_grad", _test)

In [ ]:
# @title ✅ Solution (double-click to reveal) { display-mode: "form" }
# Solutions only execute when RUN_SOLUTIONS = True (setup cell), so "Run all" stays fast.
if RUN_SOLUTIONS:
    def numerical_grad(f, w, eps=1e-6):
        g = torch.zeros_like(w)
        for i in range(w.numel()):
            w_plus, w_minus = w.clone(), w.clone()
            w_plus[i] += eps
            w_minus[i] -= eps
            g[i] = (f(w_plus) - f(w_minus)) / (2 * eps)
        return g

    f = lambda w: (w.sin() * w.exp()).sum()
    w = torch.randn(4, dtype=torch.float64, requires_grad=True)
    f(w).backward()
    num = numerical_grad(f, w.detach())
    print("autograd :", w.grad)
    print("numerical:", num)
    print("max difference:", (w.grad - num).abs().max().item())
    # PyTorch ships this check as torch.autograd.gradcheck(f, (w,)), used to test custom operations.
else:
    print("Solution not executed (RUN_SOLUTIONS = False).")

### 📌 Key takeaways
- Set `requires_grad=True` on what you want gradients for. `loss.backward()` fills `.grad` of those leaf tensors.
- `backward()` **accumulates**. Clear gradients before every step (`optimizer.zero_grad()`).
- Use `torch.no_grad()` for evaluation and for manual updates; use `.item()` to log losses.
- The graph is rebuilt every forward pass, so Python control flow just works.

### ✔️ Check yourself

Commit to an answer (say it out loud or write it down) *before* you expand it.

**1.** You call `loss.backward()` on two different mini-batches without zeroing in between, then `optimizer.step()`. What gradient does the step use?

<details>
<summary>Answer</summary>

The **sum** of the two mini-batch gradients, because `backward()` adds into `.grad`.

</details>

**2.** Does `model.eval()` stop autograd from recording? (You will meet `eval()` in Chapter 4, but you can answer from this chapter.)

<details>
<summary>Answer</summary>

No. Only `torch.no_grad()` / `torch.inference_mode()` stop recording. `eval()` is a different switch that changes the behaviour of layers like Dropout.

</details>

**3.** Why must a manual parameter update be wrapped in `torch.no_grad()`?

<details>
<summary>Answer</summary>

Otherwise the update would be recorded in the graph, and the next backward pass would differentiate through it. PyTorch forbids in-place changes to leaves that require gradients outside `no_grad`.

</details>

**4.** After `L.backward()`, which tensors have a populated `.grad`?

<details>
<summary>Answer</summary>

Only leaf tensors with `requires_grad=True`, such as parameters. Intermediate results do not keep their gradients unless you call `.retain_grad()` on them first.

</details>

**5.** Your training loop's memory use grows every iteration until it crashes. What is the first thing to check?

<details>
<summary>Answer</summary>

Whether a loss (or any tensor attached to the graph) is being accumulated for logging without `.item()` or `.detach()`.

</details>

---
# Chapter 3. Linear regression four ways: what each abstraction replaces

🎯 **In this chapter you will learn to:** map each PyTorch abstraction (autograd, optimizers, modules, loss objects) onto the manual step it replaces, and read any training loop as *forward, loss, zero_grad, backward, step*.

PyTorch code can look like magic until you see what each piece replaces. We will fit the same linear regression four times, automating one more piece each time. By the end, every training loop you read will look like "the manual algorithm, with these parts automated".

**The problem.** The data follow $y = 3x_1 - 2x_2 + 1 + \text{noise}$. The model is $\hat y = X w + b$ and the loss is the mean squared error $J = \frac{1}{N}\sum_i (\hat y_i - y_i)^2$. A good fit should recover $w \approx (3, -2)$ and $b \approx 1$.

In [ ]:
set_seed(0)
N = 200
X = torch.randn(N, 2)                                    # 200 examples, 2 features
true_w, true_b = torch.tensor([3.0, -2.0]), 1.0
y = (X @ true_w + true_b + 0.1 * torch.randn(N)).unsqueeze(1)   # shape (N, 1): see the pitfall in §1.4
print("X:", X.shape, "| y:", y.shape)

LR, STEPS = 0.1, 100
curves = {}                                              # loss history of each version, for comparison

### Version 1: everything by hand
From calculus, the gradients of $J$ are $\partial J/\partial w = \frac{2}{N} X^\top (\hat y - y)$ and $\partial J/\partial b = \frac{2}{N} \sum_i (\hat y_i - y_i)$. We code those formulas directly. No autograd is involved.

In [ ]:
w = torch.zeros(2, 1)
b = torch.zeros(1)
curves["1: by hand"] = []
for step in range(STEPS):
    y_hat = X @ w + b                       # forward pass: (N, 2) @ (2, 1) + (1,) -> (N, 1)
    err = y_hat - y
    curves["1: by hand"].append((err ** 2).mean().item())
    grad_w = 2 / N * X.T @ err              # gradient formulas, derived on paper
    grad_b = 2 / N * err.sum(dim=0)
    w -= LR * grad_w                        # update rule
    b -= LR * grad_b
print("version 1: w =", w.squeeze().tolist(), " b =", b.item())

### Version 2: autograd replaces the calculus

In [ ]:
w = torch.zeros(2, 1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)
curves["2: autograd"] = []
for step in range(STEPS):
    loss = ((X @ w + b - y) ** 2).mean()
    curves["2: autograd"].append(loss.item())
    loss.backward()                         # <- replaces the two gradient formulas
    with torch.no_grad():
        w -= LR * w.grad
        b -= LR * b.grad
    w.grad = None
    b.grad = None
print("version 2: w =", w.detach().squeeze().tolist(), " b =", b.item())

### Version 3: an optimizer replaces the update and the clearing
An **optimizer** is given the list of tensors to update. `step()` applies the update rule to each of them, and `zero_grad()` clears their gradients. For plain SGD, `step()` computes exactly `p -= lr * p.grad` inside `no_grad`, for every parameter `p`.

In [ ]:
w = torch.zeros(2, 1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)
optimizer = torch.optim.SGD([w, b], lr=LR)     # the optimizer keeps references to w and b
curves["3: optimizer"] = []
for step in range(STEPS):
    loss = ((X @ w + b - y) ** 2).mean()
    curves["3: optimizer"].append(loss.item())
    optimizer.zero_grad()                     # <- replaces "w.grad = None; b.grad = None"
    loss.backward()
    optimizer.step()                          # <- replaces the no_grad update
print("version 3: w =", w.detach().squeeze().tolist(), " b =", b.item())

### Version 4: a module and a loss object replace the parameters and the formula
`nn.Linear(2, 1)` creates and owns the weight and bias (with `requires_grad=True`), and `model.parameters()` hands them to the optimizer. `nn.MSELoss()` is the loss formula.

In [ ]:
model = nn.Linear(2, 1)
nn.init.zeros_(model.weight)                  # start from zero, like the other versions
nn.init.zeros_(model.bias)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=LR)
curves["4: module"] = []
for step in range(STEPS):
    y_hat = model(X)                          # 1. forward
    loss = loss_fn(y_hat, y)                  # 2. loss
    optimizer.zero_grad()                     # 3. clear old gradients
    loss.backward()                           # 4. backward
    optimizer.step()                          # 5. update
    curves["4: module"].append(loss.item())
print("version 4: w =", model.weight.detach().squeeze().tolist(), " b =", model.bias.item())

In [ ]:
plt.figure(figsize=(6, 3.5))
for (name, c), ls in zip(curves.items(), ["-", "--", ":", "-."]):
    plt.semilogy(c, ls, lw=2, label=name)
plt.xlabel("step"); plt.ylabel("MSE (log scale)"); plt.legend(); plt.title("Four implementations, one algorithm")
plt.show()
print("largest difference between the loss curves:",
      max(abs(a - b) for a, b in zip(curves["1: by hand"], curves["4: module"])))

All four versions are **the same algorithm**: the curves lie on top of each other, up to floating-point rounding. Here is what each abstraction took over:

| You stopped writing… | …because this does it |
| --- | --- |
| The derivative formulas | `loss.backward()` (autograd) |
| The update rule and the gradient clearing | `optimizer.step()` and `optimizer.zero_grad()` |
| Creating parameters with the right shapes | `nn.Linear` (a module) |
| The loss formula | `nn.MSELoss` |

The payoff of this design is that changes are local. Switching from SGD to Adam changes **one line**. Replacing `nn.Linear` with a 50-layer network changes **one line**. The five-step loop stays exactly the same.

### The canonical training step
Memorise this order. You will read it in thousands of code bases:

```python
pred = model(x)              # 1. forward
loss = loss_fn(pred, y)      # 2. loss
optimizer.zero_grad()        # 3. clear old gradients
loss.backward()              # 4. compute new gradients
optimizer.step()             # 5. update parameters
```

Some code puts `zero_grad()` first, before the forward pass. That is equally correct. The only requirement is that it happens **before `backward()`** in each step.

## 🧪 Exercises

**Exercise 3.1: the silent shape bug, live.** `★☆☆ · ~10 min`
Repeat version 4, but use `y_flat = y.squeeze(1)` (shape `(N,)`) as the target. Note the warning PyTorch prints and the learned weights. Using §1.4, explain why the model fails to learn `w`.

In [ ]:
# ✍️ Your attempt here.

In [ ]:
# @title ✅ Solution (double-click to reveal) { display-mode: "form" }
# Solutions only execute when RUN_SOLUTIONS = True (setup cell), so "Run all" stays fast.
if RUN_SOLUTIONS:
    y_flat = y.squeeze(1)
    bad = nn.Linear(2, 1)
    opt_bad = torch.optim.SGD(bad.parameters(), lr=LR)
    for step in range(STEPS):
        loss = F.mse_loss(bad(X), y_flat)        # (N,1) vs (N,) -> broadcast to (N,N); PyTorch warns
        opt_bad.zero_grad(); loss.backward(); opt_bad.step()
    print("w:", bad.weight.detach().squeeze().tolist(), " b:", bad.bias.item(), " mean(y):", y.mean().item())
    # Every prediction is compared with EVERY target. The best a model can do against all targets at
    # once is to predict their mean for every input, so w -> about 0 and b -> mean(y). It learns nothing about X.
    # (The F.mse_loss warning says exactly this. A hand-written loss would not even warn you.)
else:
    print("Solution not executed (RUN_SOLUTIONS = False).")

**Exercise 3.2: change the optimizer with one line.** `★☆☆ · ~10 min`
Copy version 4 and replace `torch.optim.SGD` with `torch.optim.Adam`, keeping `lr=0.1`. Does it still reach $w \approx (3, -2)$ within 100 steps? Then try `lr=0.01`. What do you observe, and why could Adam need a different learning rate from SGD?

In [ ]:
# ✍️ Your attempt here.

In [ ]:
# @title ✅ Solution (double-click to reveal) { display-mode: "form" }
# Solutions only execute when RUN_SOLUTIONS = True (setup cell), so "Run all" stays fast.
if RUN_SOLUTIONS:
    for lr in (0.1, 0.01):
        m = nn.Linear(2, 1); nn.init.zeros_(m.weight); nn.init.zeros_(m.bias)
        opt = torch.optim.Adam(m.parameters(), lr=lr)
        for step in range(STEPS):
            loss = F.mse_loss(m(X), y)
            opt.zero_grad(); loss.backward(); opt.step()
        print(f"Adam lr={lr}: w = {[round(v, 3) for v in m.weight.detach().squeeze().tolist()]}, loss = {loss.item():.4f}")
    # Adam's step size is roughly lr in EVERY coordinate, regardless of the gradient's magnitude
    # (it divides by a running estimate of the gradient's size). With lr=0.01 and 100 steps, w can
    # move at most about 0.01 * 100 = 1 from zero, so it cannot reach 3 in time.
    # SGD's step is lr * gradient, so the two lr values mean different things. Learning rates do
    # NOT transfer between optimizers.
else:
    print("Solution not executed (RUN_SOLUTIONS = False).")

### 📌 Key takeaways
- autograd replaces derivatives; optimizers replace updates and clearing; modules own parameters; loss objects replace formulas.
- The training step is always **forward → loss → zero_grad → backward → step**.
- A learning rate is only meaningful together with its optimizer.

### ✔️ Check yourself

Commit to an answer (say it out loud or write it down) *before* you expand it.

**1.** In version 4, which object owns the parameters, and how does the optimizer find them?

<details>
<summary>Answer</summary>

The `nn.Linear` module owns `weight` and `bias`. `model.parameters()` yields them, and the optimizer stores references to those exact tensors when it is constructed. It updates them in place.

</details>

**2.** What does `optimizer.step()` compute for plain SGD, in one line?

<details>
<summary>Answer</summary>

`p -= lr * p.grad` for every parameter `p` it manages, without recording autograd history.

</details>

**3.** A colleague's loop runs `optimizer.zero_grad()`, then `optimizer.step()`, then `loss.backward()`. No error is raised. What happens to the model?

<details>
<summary>Answer</summary>

Nothing: it never changes. `zero_grad()` sets every `.grad` to `None`, so `step()` finds no gradients and skips every parameter. The gradients computed by `backward()` are then wiped by the next `zero_grad()` before any step can use them. The loss stays flat, which is the clue. The order must be backward, then step.

</details>

---
# Chapter 4. `nn.Module`: how models are built

🎯 **In this chapter you will learn to:** write your own model classes; inspect and count parameters; use `Sequential`, `ModuleList` and `ModuleDict` correctly; know exactly what `train()` and `eval()` change; and save and load weights.

In PyTorch, every layer, every model and even every loss is an **`nn.Module`**. A module does four jobs for you:

1. **Bookkeeping.** It finds every parameter inside it, however deeply nested, so that `model.parameters()`, `model.to(device)` and `model.state_dict()` all work on the whole model with one call.
2. **Computation.** You write the forward pass in a method called `forward`.
3. **Mode.** It carries a `training` flag that layers such as Dropout read (§4.5).
4. **Composition.** Modules contain modules. A model is a tree of modules, and you build big models from small ones, like functions calling functions.

## 4.1 Writing your own module
Every custom module follows the same two-method pattern. Read the comments carefully: they contain the rules.

In [ ]:
class MLP(nn.Module):
    def __init__(self, in_dim=784, hidden=256, out_dim=10):
        super().__init__()                         # ALWAYS the first line: sets up the bookkeeping
        # Create every layer that has parameters HERE, once, and assign it to an attribute.
        # Assigning a module to self.<name> is what registers it.
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, out_dim)

    def forward(self, x):                          # x: (N, 1, 28, 28) or (N, 784)
        x = x.flatten(start_dim=1)                 # -> (N, 784)
        x = F.relu(self.fc1(x))                    # -> (N, 256)
        return self.fc2(x)                         # -> (N, 10): raw scores ("logits"), see Chapter 5

model = MLP()
print(model)                                       # the registered submodules
out = model(torch.randn(4, 1, 28, 28))             # call the model itself...
print("output shape:", out.shape)                  # ...which runs forward()

**`__init__` versus `forward`.** `__init__` runs **once**, when you create the model. It builds the layers and therefore the parameters. `forward` runs on **every call**, and describes how data flows through those layers. A layer created inside `forward` would get brand-new random weights on every call and could never learn (Exercise 4.2 shows this bug).

⚠️ **Always call `model(x)`, never `model.forward(x)`.** Calling the model runs some extra machinery (such as "hooks", which you will use in Notebook 2) and then calls `forward`. Calling `forward` directly silently skips that machinery.

## 4.2 Parameters: inspecting and counting them
A fully connected layer from $n_\text{in}$ to $n_\text{out}$ units has $n_\text{in} \cdot n_\text{out}$ weights plus $n_\text{out}$ biases. For our MLP, 784 → 256 → 10:

$$(784 \cdot 256 + 256) + (256 \cdot 10 + 10) = 200{,}960 + 2{,}570 = 203{,}530.$$

In [ ]:
for name, p in model.named_parameters():
    print(f"{name:12s} shape={str(tuple(p.shape)):12s} count={p.numel():>8,d}  requires_grad={p.requires_grad}")

def count_params(m: nn.Module) -> int:
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

print(f"total trainable parameters: {count_params(model):,}")
assert count_params(model) == 203_530

The names (`fc1.weight`, `fc2.bias`, …) come from the attribute names you chose. They matter: they are the keys used when saving and loading weights (§4.8), and you will use them to treat parameter groups differently (Notebook 2).

### Your own learnable tensors: `nn.Parameter`
Sometimes a model needs a learnable number or tensor that is not part of any standard layer. Research code does this all the time, for example a learnable scale or temperature. Wrap the tensor in **`nn.Parameter`** and assign it as an attribute. A plain tensor attribute is **not** registered, so the optimizer will never update it.

In [ ]:
class ScaledLinear(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.linear = nn.Linear(d_in, d_out)
        self.scale = nn.Parameter(torch.ones(1))     # registered: learnable
        self.not_learned = torch.ones(1)             # plain tensor: invisible to PyTorch

    def forward(self, x):
        return self.scale * self.linear(x)

print([name for name, _ in ScaledLinear(3, 2).named_parameters()])   # 'not_learned' is missing

## 4.3 Containers: `Sequential`, `ModuleList`, `ModuleDict`

| Container | What it is | Use it when |
| --- | --- | --- |
| `nn.Sequential(a, b, c)` | Runs `a`, then `b`, then `c`, feeding each output into the next | The model is a simple chain (very common) |
| `nn.ModuleList([...])` | A Python list whose modules **are registered** | You need custom logic in `forward`: loops, skip connections |
| `nn.ModuleDict({...})` | A dictionary whose modules are registered | Several named alternatives, such as one output head per task |

In [ ]:
# Sequential: no forward() to write. Our MLP again, in one expression:
seq = nn.Sequential(
    nn.Flatten(),                  # module version of x.flatten(start_dim=1)
    nn.Linear(784, 256),
    nn.ReLU(),
    nn.Linear(256, 10),
)
print(seq)
print("seq[1] is the first Linear layer:", seq[1])
print("same parameter count as MLP:", count_params(seq))

You will often see `nn.Sequential(OrderedDict([("fc1", nn.Linear(...)), ...]))` in other people's code. That gives the layers readable names (`seq.fc1`) instead of numbers (`seq[1]`).

⚠️ **Pitfall: a plain Python list of layers is invisible to PyTorch.** The layers still run in `forward`, so nothing crashes. But their parameters are not registered: the optimizer never updates them, `.to(device)` does not move them, and saving the model does not save them.

In [ ]:
class Broken(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = [nn.Linear(10, 10) for _ in range(3)]                 # plain list: NOT registered

class Fixed(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(10, 10) for _ in range(3)])  # registered

    def forward(self, x):
        for layer in self.layers:                  # custom logic is fine with ModuleList
            x = F.relu(layer(x))
        return x

print("Broken parameter count:", count_params(Broken()))
print("Fixed parameter count: ", count_params(Fixed()))

## 4.4 Module or function? `nn.ReLU()` versus `F.relu`
Most operations exist in two forms: a module (`nn.ReLU()`, created in `__init__`) and a function (`F.relu(x)`, called in `forward`). The rule for choosing:

- **Anything with parameters** (`nn.Linear`, `nn.Conv2d`, `nn.BatchNorm1d`) **must be a module created in `__init__`**.
- **Anything that behaves differently in training and evaluation** (`Dropout`, `BatchNorm`) **should be a module**, so that `model.eval()` switches it automatically.
- **Stateless operations** (`relu`, `flatten`, `softmax`) can be either. `F.relu` inside `forward` is common and clean; inside `nn.Sequential` you need the module form.

## 4.5 `train()` and `eval()`: what they actually change
`model.train()` and `model.eval()` set the `training` flag on the model **and on every submodule**. Only a few layer types read the flag:

| Layer | In `train()` mode | In `eval()` mode |
| --- | --- | --- |
| `nn.Dropout(p)` | Sets each value to 0 with probability `p`, and multiplies the survivors by `1/(1−p)` so the expected value is unchanged | Does nothing (identity) |
| `nn.BatchNorm1d/2d` | Normalises with the **current batch's** mean and variance, and updates its running averages | Normalises with the stored **running averages**; updates nothing |
| Every other layer | — | — |

In [ ]:
drop = nn.Dropout(p=0.5)
x = torch.ones(1, 8)
drop.train()
print("train mode:", drop(x))      # about half the values are 0, the rest are scaled to 2.0
drop.eval()
print("eval mode: ", drop(x))      # unchanged

**BatchNorm and buffers.** BatchNorm keeps running averages of the mean and variance it has seen during training, so that at evaluation time it can normalise a single example without needing a batch. These running averages are **buffers**: tensors that belong to the module, move with `.to(device)` and are saved with the model, but are **not** parameters, so the optimizer does not touch them.

In [ ]:
bn = nn.BatchNorm1d(3)
print("parameters:", [n for n, _ in bn.named_parameters()])    # weight (gamma) and bias (beta): learned
print("buffers:   ", [n for n, _ in bn.named_buffers()])       # running statistics: not learned

bn.train()
data = torch.randn(256, 3) * 5 + 10            # features with mean ~10 and std ~5
_ = bn(data)                                   # one training-mode call updates the running statistics
print("running_mean after one batch:", bn.running_mean)   # moved 10% of the way from 0 towards ~10

You will see `self.register_buffer("name", tensor)` in research code. It creates your own buffer, for example a fixed normalisation constant or a mask that should travel with the model.

⚠️ **`eval()` does not turn off gradient recording.** The two switches are independent, and evaluation code needs **both**:

```python
model.eval()                 # switch 1: Dropout off, BatchNorm uses running statistics
with torch.no_grad():        # switch 2: do not record the graph
    predictions = model(x)
model.train()                # switch back before training continues!
```

Forgetting `model.eval()` makes evaluation results noisy (Dropout is still active). Forgetting `model.train()` afterwards silently trains without Dropout. Both are very common bugs. Chapter 7 packages both switches into a function so that you cannot forget them.

## 4.6 Moving a model to the GPU
`model.to(device)` moves every parameter and buffer, recursively. For modules (unlike tensors) it works **in place** and also returns the model, so both styles below are fine. The input batch must then be moved to the same device.

In [ ]:
model = MLP().to(device)
print("model is on:", next(model.parameters()).device)     # check where a model lives
x = torch.randn(4, 1, 28, 28).to(device)                   # the data must follow
print("output:", model(x).shape, "on", model(x).device)

## 4.7 Initialisation
Every layer initialises its own parameters when it is created. `nn.Linear` uses a uniform distribution scaled by $1/\sqrt{n_\text{in}}$, which works well for the shallow networks in this notebook. Papers sometimes require a specific scheme (for example He initialisation for deep ReLU networks). The idiom is to write a function that initialises one module, then use **`model.apply(fn)`**, which calls `fn` on every submodule recursively:

In [ ]:
def init_he(m):
    if isinstance(m, nn.Linear):                              # only touch Linear layers
        nn.init.kaiming_normal_(m.weight, nonlinearity="relu") # He initialisation
        nn.init.zeros_(m.bias)

model = MLP()
model.apply(init_he)
print(f"fc1 weight std: {model.fc1.weight.std():.4f} (He target: sqrt(2/784) = {math.sqrt(2 / 784):.4f})")

## 4.8 Saving and loading weights: `state_dict`
`model.state_dict()` is a dictionary from names to tensors that holds **every parameter and buffer**. It does **not** contain the code, so it does not describe the architecture. To load weights, you first build the same architecture in code, then load the dictionary into it.

In [ ]:
model = MLP()
sd = model.state_dict()
print({k: tuple(v.shape) for k, v in sd.items()})

path = DATA_DIR / "mlp_weights.pt"
torch.save(sd, path)                                    # save the weights to a file

fresh = MLP()                                           # 1. rebuild the architecture in code
fresh.load_state_dict(torch.load(path, map_location="cpu"))   # 2. load the weights into it
x = torch.randn(2, 784)
print("same outputs after loading:", torch.allclose(model(x), fresh(x)))

`map_location="cpu"` lets you load weights saved on a GPU machine onto a machine without one. Recent PyTorch versions also load in a safe mode by default (`weights_only=True`), which refuses to run arbitrary code hidden in a file. Notebook 2 builds full training checkpoints on top of this.

Loading into a **different** architecture fails loudly, which is what you want:

In [ ]:
try:
    MLP(hidden=128).load_state_dict(sd)                 # different hidden size
except RuntimeError as e:
    print("RuntimeError:", str(e)[:230], "...")

## 📖 Code-reading drill
This block is typical of code you will find in papers' repositories. Read it and answer **without running it**:

```python
class Block(nn.Module):
    def __init__(self, dim, p=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.ff = nn.Sequential(nn.Linear(dim, 4 * dim), nn.GELU(), nn.Linear(4 * dim, dim))
        self.drop = nn.Dropout(p)
        self.gamma = nn.Parameter(1e-4 * torch.ones(dim))

    def forward(self, x):                      # x: (N, dim)
        return x + self.gamma * self.drop(self.ff(self.norm(x)))
```

1. What is the output shape for an input of shape `(32, 64)` with `dim=64`?
2. How many trainable parameters does `Block(64)` have? (`nn.LayerNorm(dim)` has `2·dim`.)
3. Which parts behave differently in `eval()` mode?
4. At the start of training, is the block close to the identity function? Why?

In [ ]:
class Block(nn.Module):
    def __init__(self, dim, p=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.ff = nn.Sequential(nn.Linear(dim, 4 * dim), nn.GELU(), nn.Linear(4 * dim, dim))
        self.drop = nn.Dropout(p)
        self.gamma = nn.Parameter(1e-4 * torch.ones(dim))

    def forward(self, x):
        return x + self.gamma * self.drop(self.ff(self.norm(x)))

blk = Block(64)
print("1.", blk(torch.randn(32, 64)).shape)
print("2.", count_params(blk), "=", 2 * 64, "+", 64 * 256 + 256, "+", 256 * 64 + 64, "+", 64)
# 3. Only the Dropout layer. (LayerNorm, unlike BatchNorm, behaves the same in both modes.)
# 4. Yes: gamma starts at 1e-4, so the output is x plus a tiny correction. This "residual
#    connection with a small learnable scale" makes very deep stacks of blocks easy to train.

## 🧪 Exercises

**Exercise 4.1: a configurable MLP factory.** `★★☆ · ~20 min`
Write `make_mlp(in_dim, hidden_dims, out_dim, p_drop=0.0, batchnorm=False)` that returns an `nn.Sequential`. It should start with `nn.Flatten()`. Then, for each size `h` in `hidden_dims`, add `Linear → (BatchNorm1d if batchnorm) → ReLU → (Dropout if p_drop > 0)`. Finish with a `Linear` to `out_dim` and **no** activation. Then verify by hand that `make_mlp(784, [512, 256], 10, batchnorm=True)` has 537,354 parameters.

In [ ]:
# ✍️ Your attempt here.
# Hint: Build a Python list `layers`, append to it in a loop, and return nn.Sequential(*layers).

In [ ]:
# 🔎 Self-check for Exercise 4.1
def _test(fn):
    m = fn(784, [512, 256], 10, batchnorm=True)
    assert isinstance(m, nn.Sequential), "return an nn.Sequential"
    n = sum(p.numel() for p in m.parameters())
    assert n == 537_354, f"{n:,} parameters, expected 537,354"
    assert isinstance(m[-1], nn.Linear), "the last layer must be a Linear layer (no activation after it)"
    assert fn(784, [64], 10, p_drop=0.5)(torch.randn(4, 1, 28, 28)).shape == (4, 10), "wrong output shape"
    assert any(isinstance(l, nn.Dropout) for l in fn(784, [64], 10, p_drop=0.5)), "Dropout missing"
self_check("make_mlp", _test)

In [ ]:
# @title ✅ Solution (double-click to reveal) { display-mode: "form" }
# Solutions only execute when RUN_SOLUTIONS = True (setup cell), so "Run all" stays fast.
if RUN_SOLUTIONS:
    def make_mlp(in_dim, hidden_dims, out_dim, p_drop=0.0, batchnorm=False):
        layers = [nn.Flatten()]
        d = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(d, h))
            if batchnorm:
                layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            if p_drop > 0:
                layers.append(nn.Dropout(p_drop))
            d = h
        layers.append(nn.Linear(d, out_dim))
        return nn.Sequential(*layers)

    by_hand = (784 * 512 + 512) + 2 * 512 + (512 * 256 + 256) + 2 * 256 + (256 * 10 + 10)
    print(f"by hand: {by_hand:,}   counted: {count_params(make_mlp(784, [512, 256], 10, batchnorm=True)):,}")
else:
    print("Solution not executed (RUN_SOLUTIONS = False).")

**Exercise 4.2: find the bugs.** `★★☆ · ~20 min`
The module below runs without any error, yet it has **three** bugs that would ruin training. Run the diagnostic cell, then find and fix all three in a class called `FixedNet`.

In [ ]:
class BuggyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = [nn.Linear(784, 256), nn.Linear(256, 10)]

    def forward(self, x):
        x = x.flatten(start_dim=1)
        x = F.relu(self.layers[0](x))
        x = F.dropout(x, p=0.5)
        x = self.layers[1](x)
        return nn.Linear(10, 10)(x)

net = BuggyNet()
net.eval()
x = torch.randn(2, 784)
print("trainable parameters:", count_params(net))
print("two eval-mode calls on the same input agree?", torch.allclose(net(x), net(x)))

In [ ]:
# ✍️ Your attempt here.
# Hint: Check where parameters are registered, whether dropout knows about eval mode, and which layers are created on every call.

In [ ]:
# @title ✅ Solution (double-click to reveal) { display-mode: "form" }
# Solutions only execute when RUN_SOLUTIONS = True (setup cell), so "Run all" stays fast.
if RUN_SOLUTIONS:
    class FixedNet(nn.Module):
        def __init__(self):
            super().__init__()
            # Bug 1: a plain list hides the layers from PyTorch -> use nn.ModuleList (or named attributes).
            self.layers = nn.ModuleList([nn.Linear(784, 256), nn.Linear(256, 10)])
            # Bug 3 (fix part): create the extra layer ONCE, here.
            self.out = nn.Linear(10, 10)
            # Bug 2 (fix part): a Dropout MODULE follows model.train()/eval() automatically.
            self.drop = nn.Dropout(0.5)

        def forward(self, x):
            x = x.flatten(start_dim=1)
            x = F.relu(self.layers[0](x))
            x = self.drop(x)       # Bug 2: F.dropout defaults to training=True, so it stayed on in eval mode
            x = self.layers[1](x)
            return self.out(x)     # Bug 3: nn.Linear(10, 10) inside forward = new random weights every call

    net = FixedNet(); net.eval()
    x = torch.randn(2, 784)
    print("trainable parameters:", count_params(net))
    print("two eval-mode calls agree?", torch.allclose(net(x), net(x)))
else:
    print("Solution not executed (RUN_SOLUTIONS = False).")

### 📌 Key takeaways
- Create layers in `__init__`, use them in `forward`, call `model(x)`.
- Register submodules as attributes or in `ModuleList`/`ModuleDict`; use `nn.Parameter` for custom learnable tensors.
- `eval()` switches Dropout and BatchNorm. `no_grad()` switches off gradient recording. You need both for evaluation, and `train()` afterwards.
- `state_dict()` holds the weights, not the architecture: rebuild the model in code, then load.

### ✔️ Check yourself

Commit to an answer (say it out loud or write it down) *before* you expand it.

**1.** Your optimizer reports far fewer parameters than expected, and some layers never change during training. Name two likely causes.

<details>
<summary>Answer</summary>

The layers are stored in a plain Python list or dict instead of `nn.ModuleList`/`nn.ModuleDict`, or they are created inside `forward` instead of `__init__`.

</details>

**2.** What is the difference between a parameter and a buffer?

<details>
<summary>Answer</summary>

Both are saved in the `state_dict` and moved by `.to(device)`. Parameters are returned by `.parameters()` and updated by the optimizer; buffers are not (for example BatchNorm's running mean).

</details>

**3.** You evaluate a model with Dropout twice on the same validation set and get different accuracies. What did you forget?

<details>
<summary>Answer</summary>

`model.eval()`. In training mode, Dropout randomly zeroes activations on every call.

</details>

**4.** Why can you not load a `state_dict` without the model's code?

<details>
<summary>Answer</summary>

The `state_dict` only stores named tensors. The architecture (which layers exist and how `forward` connects them) lives in the Python class, which must be instantiated first.

</details>

---
# Chapter 5. Loss functions and the "logits rule"

🎯 **In this chapter you will learn to:** choose the right loss for regression, binary, multi-class and multi-label problems; give it outputs and targets with the right shapes and dtypes; and avoid the two silent bugs that ruin classification models.

You know from your course *why* mean squared error suits regression and cross-entropy suits classification. In PyTorch, the difficulty is the **contract** of each loss: what shape and dtype it expects for the model output and for the target. Most loss bugs are contract violations, and several of them raise no error.

## 5.1 The table you will use daily

| Task | Model's last layer outputs | Loss | Target: dtype and shape | Prediction |
| --- | --- | --- | --- | --- |
| Regression | `(N, 1)` real values | `nn.MSELoss` (or `nn.L1Loss`, `nn.HuberLoss`) | `float32`, `(N, 1)` | the output itself |
| Binary classification | `(N, 1)` **logits** | `nn.BCEWithLogitsLoss` | `float32` 0. or 1., `(N, 1)` | `logit > 0` |
| Multi-class (one label per example) | `(N, C)` **logits** | `nn.CrossEntropyLoss` | **`int64` class indices, `(N,)`** | `argmax(dim=1)` |
| Multi-label (several labels may be true) | `(N, C)` logits | `nn.BCEWithLogitsLoss` | `float32` 0./1., `(N, C)` | `logits > 0`, per class |

**Logits** are the raw, unnormalised scores produced by the last `Linear` layer, before any sigmoid or softmax. They can be any real number.

> **The logits rule.** Your model outputs logits. The loss applies the sigmoid or softmax **internally**, in a numerically stable combined form. You apply `softmax` or `sigmoid` yourself **only** when you need actual probabilities, for example to report confidence, and never before the loss.

## 5.2 `CrossEntropyLoss` opened up
`nn.CrossEntropyLoss` combines two steps: a `log_softmax` over the classes, then the **negative log-likelihood** of the correct class, averaged over the batch. Let us verify that on four examples.

In [ ]:
logits = torch.tensor([[2.0, 1.0, 0.1],      # 4 examples, 3 classes
                       [0.5, 2.5, 0.3],
                       [1.2, 0.2, 3.1],
                       [0.1, 0.1, 0.1]])
targets = torch.tensor([0, 1, 2, 0])          # int64 class indices, shape (4,)

loss = nn.CrossEntropyLoss()(logits, targets)
print(f"CrossEntropyLoss: {loss.item():.4f}")

log_p = torch.log_softmax(logits, dim=1)                   # log-probabilities, (4, 3)
per_example = -log_p[torch.arange(4), targets]             # -log p(correct class), the pattern of §1.3
print("per-example losses:", per_example)
print(f"their mean:       {per_example.mean().item():.4f}")
print(f"last example: {per_example[3].item():.4f} = ln 3 = {math.log(3):.4f} (all logits equal -> p = 1/3)")

In older code you will see `nn.NLLLoss()` applied to `F.log_softmax(logits, dim=1)`. That is the same thing in two steps, and it is correct. What is **not** correct is the next pattern.

Turning logits into predictions and probabilities:

In [ ]:
predicted_class = logits.argmax(dim=1)            # no softmax needed: softmax does not change the order
probabilities = torch.softmax(logits, dim=1)      # only when you want to REPORT probabilities
print("predicted:", predicted_class)
print("probabilities (each row sums to 1):\n", probabilities.round(decimals=3))

## 5.3 Silent bug #1: a softmax before `CrossEntropyLoss`
If the model ends with a softmax and you then use `CrossEntropyLoss`, softmax is applied **twice**. No error appears. But the "logits" the loss sees are now probabilities between 0 and 1, so the loss can never get close to 0, and the gradients become small and distorted. The model trains slower and usually ends up worse, and nothing tells you why.

In [ ]:
confident = torch.tensor([[10.0, -10.0, -10.0]])        # a very confident, CORRECT prediction
t = torch.tensor([0])
print("correct use (logits in):     loss =", round(F.cross_entropy(confident, t).item(), 5))
print("bug (softmax, then the loss): loss =", round(F.cross_entropy(torch.softmax(confident, dim=1), t).item(), 5))

Even a perfect prediction costs 0.55 under the double softmax: the smallest possible value is $-\log\frac{e^1}{e^1 + 2e^0} \approx 0.55$. The network keeps receiving gradient for predictions that are already right.

## 5.4 Target formats and the errors they produce
`CrossEntropyLoss` is strict about its target:
- **`int64` class indices, shape `(N,)`**: the standard format.
- **`float32` class probabilities, shape `(N, C)`**: also accepted. It is used for label smoothing and knowledge distillation. A float one-hot matrix gives the same loss as the indices.
- Other combinations raise errors. Learn to recognise them:

In [ ]:
print("int64 indices:     ", F.cross_entropy(logits, targets).item())
print("float one-hot:     ", F.cross_entropy(logits, F.one_hot(targets, 3).float()).item())

for desc, bad_target in [("float indices", targets.float()),
                         ("int64 one-hot", F.one_hot(targets, 3)),
                         ("label 3 with only 3 classes", torch.tensor([0, 1, 3, 0]))]:
    try:
        F.cross_entropy(logits, bad_target)
    except (RuntimeError, IndexError) as e:
        print(f"{desc:28s} -> {type(e).__name__}: {str(e)[:80]}")

⚠️ **On the GPU, an out-of-range label does not give a readable error.** You get `CUDA error: device-side assert triggered`, often reported at a *later* line, because GPU errors are asynchronous. After that, the GPU state is broken and you must restart the runtime. When you see this message, the first suspect is a label outside `[0, C)`, for example labels 1–10 instead of 0–9. Check with `labels.min(), labels.max()`, or rerun the code on the CPU to get a readable error.

## 5.5 Binary classification: why `BCEWithLogitsLoss`
For binary problems, the model outputs **one** logit per example, shape `(N, 1)`, and the target is a **float** 0. or 1. of the same shape. You could write `nn.BCELoss()(torch.sigmoid(logits), y)`, but that version breaks down when the model is confidently wrong. The cell below shows a positive example ($y = 1$) that the model scores at $z = -200$:

In [ ]:
z = torch.tensor([[-200.0]], requires_grad=True)
y_bin = torch.tensor([[1.0]])
unfused = F.binary_cross_entropy(torch.sigmoid(z), y_bin)
unfused.backward()
print(f"sigmoid + BCELoss : loss = {unfused.item():7.2f}, gradient = {z.grad.item():+.2f}")

z = torch.tensor([[-200.0]], requires_grad=True)
fused = F.binary_cross_entropy_with_logits(z, y_bin)
fused.backward()
print(f"BCEWithLogitsLoss : loss = {fused.item():7.2f}, gradient = {z.grad.item():+.2f}")

The true loss is $-\log \sigma(-200) = \log(1 + e^{200}) \approx 200$, and the true gradient is $\sigma(z) - y \approx -1$: a strong push in the right direction. In `float32`, $\sigma(-200)$ rounds to exactly 0, so the unfused version clips the loss to 100 and loses the gradient entirely. The fused version computes the same quantity with an algebraically rearranged, stable formula. **Same mathematics, different numerics.** That is the whole reason for the logits rule.

## 5.6 Options you will actually use

In [ ]:
logits = torch.randn(6, 3)
targets = torch.tensor([0, 1, 2, 2, 1, 0])

# Class imbalance: make mistakes on rare classes cost more
print("weighted:        ", nn.CrossEntropyLoss(weight=torch.tensor([1.0, 1.0, 5.0]))(logits, targets).item())
# Label smoothing: train towards 0.9 / 0.05 / 0.05 instead of 1 / 0 / 0 (a mild regulariser)
print("label smoothing: ", nn.CrossEntropyLoss(label_smoothing=0.1)(logits, targets).item())
# Per-example losses instead of the mean: find the hardest examples
print("per example:     ", nn.CrossEntropyLoss(reduction="none")(logits, targets))
# Binary imbalance: pos_weight = (#negatives / #positives) up-weights the positive class
bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([4.0]))
print("pos_weight BCE:  ", bce(torch.randn(6, 1), torch.randint(0, 2, (6, 1)).float()).item())

## 🧪 Exercises

**Exercise 5.1: cross-entropy from scratch.** `★★☆ · ~15 min`
Implement `my_cross_entropy(logits, targets)` for `(N, C)` logits and `(N,)` integer targets, returning the mean loss, **without** using `softmax`, `log_softmax` or any loss function. Use the identity $\log p_c = z_c - \log\sum_j e^{z_j}$ and `torch.logsumexp`, which computes $\log\sum_j e^{z_j}$ stably. It must stay finite for logits like 1000.

In [ ]:
# ✍️ Your attempt here.
# Hint: torch.logsumexp(logits, dim=1, keepdim=True) has shape (N, 1). Then pick the correct column per row as in §1.3.

In [ ]:
# 🔎 Self-check for Exercise 5.1
def _test(fn):
    lg, tg = torch.randn(16, 7), torch.randint(0, 7, (16,))
    assert torch.allclose(fn(lg, tg), F.cross_entropy(lg, tg), atol=1e-5), "does not match F.cross_entropy"
    big = torch.tensor([[1000.0, 0.0], [0.0, 1000.0]])
    out = fn(big, torch.tensor([1, 1]))
    assert torch.isfinite(out), "not numerically stable for large logits"
    assert abs(out.item() - 500.0) < 1e-3, "wrong value for large logits"
self_check("my_cross_entropy", _test)

In [ ]:
# @title ✅ Solution (double-click to reveal) { display-mode: "form" }
# Solutions only execute when RUN_SOLUTIONS = True (setup cell), so "Run all" stays fast.
if RUN_SOLUTIONS:
    def my_cross_entropy(logits, targets):
        log_p = logits - torch.logsumexp(logits, dim=1, keepdim=True)   # log-softmax, stably
        return -log_p[torch.arange(len(targets)), targets].mean()

    print(my_cross_entropy(torch.tensor([[1000.0, 0.0]]), torch.tensor([1])))   # 1000, finite
else:
    print("Solution not executed (RUN_SOLUTIONS = False).")

**Exercise 5.2: choose the setup.** `★☆☆ · ~10 min`
For each problem, state the shape of the model's output, the loss, and the target's dtype and shape (batch size `N`):
1. Predict the price of a house from 20 features.
2. Decide whether an email is spam.
3. Classify a photo as one of 37 dog breeds.
4. Tag a news article with any subset of 12 topics.

In [ ]:
# @title ✅ Solution (double-click to reveal) { display-mode: "form" }
# Solutions only execute when RUN_SOLUTIONS = True (setup cell), so "Run all" stays fast.
if RUN_SOLUTIONS:
    print('''
    1. Regression:  output (N, 1); nn.MSELoss (or HuberLoss if there are outliers); target float32 (N, 1).
    2. Binary:      output (N, 1) logits; nn.BCEWithLogitsLoss; target float32 0./1. of shape (N, 1).
    3. Multi-class: output (N, 37) logits; nn.CrossEntropyLoss; target int64 (N,) with values 0..36.
    4. Multi-label: output (N, 12) logits; nn.BCEWithLogitsLoss; target float32 multi-hot (N, 12).
    ''')
else:
    print("Solution not executed (RUN_SOLUTIONS = False).")

### 📌 Key takeaways
- Models output **logits**. Losses apply sigmoid/softmax internally. Never put a softmax before `CrossEntropyLoss`.
- `CrossEntropyLoss` wants `(N, C)` logits and `(N,)` `int64` targets. `BCEWithLogitsLoss` wants float targets with the same shape as the logits.
- `device-side assert triggered` usually means a label outside `[0, C)`.

### ✔️ Check yourself

Commit to an answer (say it out loud or write it down) *before* you expand it.

**1.** Your 10-class model's training loss plateaus at about 1.46 and never gets lower, even when accuracy is high. What do you suspect?

<details>
<summary>Answer</summary>

A softmax before `CrossEntropyLoss`. With probabilities as inputs, the smallest achievable loss for 10 classes is $-\log\frac{e}{e + 9} \approx 1.46$.

</details>

**2.** Why do you not need a softmax to compute predicted classes?

<details>
<summary>Answer</summary>

Softmax is monotonic within a row: the largest logit always has the largest probability. `argmax` of the logits gives the same class.

</details>

**3.** Your labels come from a CSV as 1, 2, …, 10. What goes wrong with `CrossEntropyLoss` on 10 classes, and how do you fix it?

<details>
<summary>Answer</summary>

Label 10 is out of range (valid labels are 0–9). On the CPU you get an `IndexError`, on the GPU a `device-side assert`. Subtract 1 from the labels.

</details>

---
# Chapter 6. Data: `Dataset`, `DataLoader` and transforms

🎯 **In this chapter you will learn to:** wrap any data in a `Dataset`; batch and shuffle it with a `DataLoader`; split data into training and validation sets without leaking information; and load and preprocess an image dataset with `torchvision`.

So far we have pushed the **whole** dataset through the model at every step. Real training uses **mini-batches**: at each step the model sees a random handful of examples (typically 32–512), computes the loss on them, and takes one gradient step. Mini-batches make each step cheap, let datasets far larger than memory be used, and the noise they add to the gradient often helps generalisation.

PyTorch splits the job of feeding data into two objects with clearly separated responsibilities:

| Object | Responsibility | Question it answers |
| --- | --- | --- |
| `Dataset` | Knows how many examples there are and how to produce example number `i`, preprocessed | "What is example 17?" |
| `DataLoader` | Picks indices (shuffled or not), asks the `Dataset` for those examples, and stacks them into a batch | "Give me the next batch." |

## 6.1 The `Dataset` protocol
A `Dataset` is any object with two methods:
- `__len__(self)`: the number of examples;
- `__getitem__(self, i)`: example `i`, usually as a tuple `(input, target)`.

When your data is already in tensors, `TensorDataset` does this for you:

In [ ]:
from torch.utils.data import Dataset, TensorDataset, DataLoader, Subset

X = torch.randn(10, 3)                     # 10 examples, 3 features
y = torch.arange(10)                       # one label per example
ds = TensorDataset(X, y)                   # pairs them up, row by row
print("len(ds) =", len(ds))
x5, y5 = ds[5]                             # one example: a (features, label) tuple
print("ds[5] ->", x5, y5)

### A custom `Dataset`: predictive maintenance
Most real data needs a custom class. As an engineering example, suppose each example is a machine with four sensor readings (temperature in °C, vibration in mm/s, pressure in bar, running hours), and the label says whether the machine failed within a month. The readings live on very different scales, so the dataset should standardise them, and the class is where that logic belongs.

In [ ]:
def make_sensor_data(n, seed=0):
    # Synthetic data: failure risk grows with temperature and vibration.
    g = torch.Generator().manual_seed(seed)
    temp = 60 + 15 * torch.randn(n, generator=g)
    vib = 3 + 1.5 * torch.randn(n, generator=g).abs()
    pres = 8 + 0.5 * torch.randn(n, generator=g)
    hours = 5000 * torch.rand(n, generator=g)
    X = torch.stack([temp, vib, pres, hours], dim=1)
    risk = 0.08 * (temp - 60) + 1.2 * (vib - 4) + 0.0002 * (hours - 2500)
    y = (risk + 0.5 * torch.randn(n, generator=g) > 0.5).long()
    return X, y

class SensorDataset(Dataset):
    def __init__(self, X, y, mean, std):
        self.X = (X - mean) / std          # standardise ONCE, with statistics we are given
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.X[i], self.y[i]

X_all, y_all = make_sensor_data(1000)
print("raw feature means:", X_all.mean(dim=0).round(decimals=1))    # wildly different scales
print("fraction of failures:", y_all.float().mean().item())

Notice that `SensorDataset` receives `mean` and `std` instead of computing them itself. That design choice prevents **data leakage**, the subject of the next section.

## 6.2 Splitting the data, and avoiding leakage
You need three disjoint sets:

| Split | Used for | Rule |
| --- | --- | --- |
| **Training** | Computing gradients | |
| **Validation** | Choosing hyper-parameters, the number of epochs, and the best checkpoint | Never train on it |
| **Test** | One final, unbiased estimate of performance | Use it **once**, at the very end |

**Data leakage** means that information from the validation or test data influences training. Leakage makes results look better than they will be in reality. A subtle and very common form is computing preprocessing statistics, such as the mean and standard deviation for standardisation, on the **whole** dataset before splitting. Those statistics then contain information about the validation and test examples.

**The rule: fit every preprocessing step on the training split only, then apply the same, frozen statistics everywhere.**

In [ ]:
# 1. Split the INDICES reproducibly: a seeded generator gives the same split on every run.
g = torch.Generator().manual_seed(0)
perm = torch.randperm(len(y_all), generator=g)
train_idx, val_idx = perm[:800], perm[800:]

# 2. Fit preprocessing on the TRAINING split only.
mean = X_all[train_idx].mean(dim=0)
std = X_all[train_idx].std(dim=0)

# 3. Apply the same statistics to every split.
train_ds = SensorDataset(X_all[train_idx], y_all[train_idx], mean, std)
val_ds = SensorDataset(X_all[val_idx], y_all[val_idx], mean, std)
print(len(train_ds), "training and", len(val_ds), "validation examples")
print("standardised training means:", train_ds.X.mean(dim=0).round(decimals=3))

`torch.utils.data.random_split(ds, [0.8, 0.2], generator=g)` is a shortcut that returns two `Subset` objects, and you will see it often. Splitting the indices yourself, as above, makes it easier to compute training-only statistics.

🏭 **Industry note.** The test set is the most valuable data you have, because once you have looked at test results and changed something because of them, the test set is no longer an unbiased estimate. Teams often lock it away until the end of a project.

## 6.3 The `DataLoader`

In [ ]:
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False)

xb, yb = next(iter(train_loader))            # fetch one batch
print("one batch:", xb.shape, yb.shape, yb.dtype)
print("batches per epoch:", len(train_loader), "= ceil(800 / 64)")
sizes = [len(yb) for _, yb in train_loader]
print("batch sizes in one epoch:", sizes)     # the last batch is smaller: 800 = 12 * 64 + 32

The `DataLoader` asked the dataset for 64 examples and **stacked** them along a new first dimension: 64 tensors of shape `(4,)` became one tensor of shape `(64, 4)`. That is why every batch has the batch dimension first.

The arguments you need to know:

| Argument | Meaning | Typical choice |
| --- | --- | --- |
| `batch_size` | Examples per batch | 32–512 for training; larger for evaluation (no gradients, so less memory) |
| `shuffle` | Reshuffle the order at the start of every epoch | `True` for training, `False` for validation and test |
| `drop_last` | Drop the final, smaller batch | Usually `False`; `True` if a small batch would cause problems (e.g. BatchNorm with batch size 1) |
| `num_workers` | Background processes that prepare batches in parallel with training | 2–8 when per-example preprocessing is slow; 0 = load in the main process |
| `pin_memory` | Put batches in page-locked memory, which speeds up copying to an NVIDIA GPU | `True` when training on CUDA |

🤔 **Predict:** with `shuffle=True`, are the first labels of the first batch the same in two different epochs? Run the cell to find out.

In [ ]:
for epoch in range(2):
    _, yb = next(iter(train_loader))
    print(f"epoch {epoch}: first 10 labels of the first batch: {yb[:10].tolist()}")

## 6.4 Image data with `torchvision`
`torchvision.datasets` downloads and wraps standard datasets. We use **Fashion-MNIST**: 70,000 grayscale 28×28 images of clothing in 10 classes, split into 60,000 training and 10,000 test images. It is a drop-in replacement for the handwritten-digit MNIST, but harder, so the differences between models are easier to see.

Look at what one raw example is **before** any transform:

In [ ]:
from torchvision import datasets
from torchvision.transforms import v2 as T       # the current transforms API

raw_train = datasets.FashionMNIST(DATA_DIR, train=True, download=True)   # no transform yet
CLASSES = raw_train.classes
img, label = raw_train[0]
print("type:", type(img).__name__, "| size:", img.size, "| mode:", img.mode)   # a PIL image
print("label:", label, "->", CLASSES[label])
print("all images as one uint8 tensor:", raw_train.data.shape, raw_train.data.dtype)

A model needs a `float32` tensor of shape `(1, 28, 28)`, ideally standardised. A **transform** is a function applied to each example inside `__getitem__`, and `T.Compose` chains several:

```python
T.Compose([
    T.ToImage(),                           # PIL image -> uint8 tensor, shape (C, H, W)
    T.ToDtype(torch.float32, scale=True),  # -> float32, and divide by 255 to get [0, 1]
    T.Normalize(mean=[m], std=[s]),        # -> (x - m) / s, per channel
])
```

📖 In older code (and most tutorials online) you will see `transforms.ToTensor()` instead of the first two lines. It does the same conversion in one step. Both versions produce identical tensors.

Following §6.2, we split **first** and compute the normalisation statistics on the training split only. Fashion-MNIST has no official validation set, so we hold out 5,000 of the 60,000 training images.

In [ ]:
g = torch.Generator().manual_seed(0)
perm = torch.randperm(len(raw_train), generator=g)
fm_val_idx, fm_train_idx = perm[:5000], perm[5000:]           # 5,000 validation, 55,000 training

train_pixels = raw_train.data[fm_train_idx].float() / 255     # (55000, 28, 28), training split only
FM_MEAN, FM_STD = train_pixels.mean().item(), train_pixels.std().item()
print(f"training-split pixel mean = {FM_MEAN:.4f}, std = {FM_STD:.4f}")

fm_transform = T.Compose([
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
    T.Normalize(mean=[FM_MEAN], std=[FM_STD]),
])
x, label = datasets.FashionMNIST(DATA_DIR, train=True, transform=fm_transform)[0]
print("after the transform:", x.shape, x.dtype, f"| mean {x.mean():.2f}, std {x.std():.2f}")

**Always look at your data.** It is the cheapest bug detector there is: wrong labels, rotated images and broken normalisation are all visible at a glance.

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(12, 3.4))
for ax, i in zip(axes.flat, range(16)):
    img, label = raw_train[i]
    ax.imshow(img, cmap="gray"); ax.set_title(CLASSES[label], fontsize=8); ax.axis("off")
plt.suptitle("Fashion-MNIST training examples"); plt.tight_layout(); plt.show()

## 🧪 Exercises

**Exercise 6.1: statistics over a loader.** `★★☆ · ~15 min`
Datasets are often too big to load at once, so statistics have to be computed batch by batch. Write `channel_stats(loader)` that takes a `DataLoader` yielding `(x, y)` with `x` of shape `(B, C, H, W)` and returns two tensors of shape `(C,)`: the per-channel mean and standard deviation over **all** pixels of all images. Accumulate sums; do not concatenate the batches.

In [ ]:
# ✍️ Your attempt here.
# Hint: Accumulate s1 = Σx and s2 = Σx² per channel, with x.sum(dim=(0, 2, 3)), plus the pixel count. Then var = E[x²] − E[x]².

In [ ]:
# 🔎 Self-check for Exercise 6.1
def _test(fn):
    data = torch.rand(100, 3, 8, 8) * torch.tensor([1.0, 2.0, 3.0]).view(1, 3, 1, 1)
    loader = DataLoader(TensorDataset(data, torch.zeros(100)), batch_size=32)
    m, s = fn(loader)
    assert m.shape == (3,) and s.shape == (3,), "return two tensors of shape (C,)"
    ref = data.transpose(0, 1).reshape(3, -1)
    assert torch.allclose(m, ref.mean(dim=1), atol=1e-4), "means are wrong"
    assert torch.allclose(s, ref.std(dim=1, unbiased=False), atol=1e-3), "standard deviations are wrong"
self_check("channel_stats", _test)

In [ ]:
# @title ✅ Solution (double-click to reveal) { display-mode: "form" }
# Solutions only execute when RUN_SOLUTIONS = True (setup cell), so "Run all" stays fast.
if RUN_SOLUTIONS:
    def channel_stats(loader):
        s1, s2, count = 0.0, 0.0, 0
        for x, _ in loader:
            x = x.double()                                   # double precision: sums of many squares
            s1 = s1 + x.sum(dim=(0, 2, 3))
            s2 = s2 + (x ** 2).sum(dim=(0, 2, 3))
            count += x.shape[0] * x.shape[2] * x.shape[3]    # pixels per channel
        mean = s1 / count
        std = (s2 / count - mean ** 2).sqrt()
        return mean.float(), std.float()

    to_tensor = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)])
    fm_plain = Subset(datasets.FashionMNIST(DATA_DIR, train=True, transform=to_tensor), fm_train_idx[:5000].tolist())
    print(channel_stats(DataLoader(fm_plain, batch_size=500)), "| full training split:", round(FM_MEAN, 4), round(FM_STD, 4))
else:
    print("Solution not executed (RUN_SOLUTIONS = False).")

**Exercise 6.2: train on the sensor data.** `★★☆ · ~20 min`
Using `train_loader` and `val_loader` from §6.3, train an MLP `4 → 16 → 2` (with ReLU) with `CrossEntropyLoss` and `torch.optim.Adam(lr=1e-2)` for 30 epochs **using mini-batches**. After each epoch, compute the validation accuracy inside `torch.no_grad()`. What accuracy does it reach? What would a model that always predicts "no failure" score (the **majority-class baseline**)?

In [ ]:
# ✍️ Your attempt here.
# Hint: Two nested loops: `for epoch in range(30):` and `for xb, yb in train_loader:` with the five-step training step inside.

In [ ]:
# @title ✅ Solution (double-click to reveal) { display-mode: "form" }
# Solutions only execute when RUN_SOLUTIONS = True (setup cell), so "Run all" stays fast.
if RUN_SOLUTIONS:
    set_seed(0)
    sensor_model = nn.Sequential(nn.Linear(4, 16), nn.ReLU(), nn.Linear(16, 2))
    opt = torch.optim.Adam(sensor_model.parameters(), lr=1e-2)
    for epoch in range(30):
        sensor_model.train()
        for xb, yb in train_loader:
            loss = F.cross_entropy(sensor_model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
        sensor_model.eval()
        with torch.no_grad():
            correct = sum((sensor_model(xb).argmax(dim=1) == yb).sum().item() for xb, yb in val_loader)
        if epoch % 10 == 9:
            print(f"epoch {epoch + 1}: validation accuracy {correct / len(val_ds):.3f}")
    majority = max(val_ds.y.float().mean().item(), 1 - val_ds.y.float().mean().item())
    print(f"majority-class baseline: {majority:.3f}")
    # Always compare with this baseline: on imbalanced data, a high accuracy can mean very little.
else:
    print("Solution not executed (RUN_SOLUTIONS = False).")

### 📌 Key takeaways
- `Dataset` = how to get example `i`. `DataLoader` = shuffling, batching (stacking along a new dimension 0) and parallel loading.
- Split first, with a seeded generator. Fit preprocessing statistics on the training split only.
- Use the test set once, at the end.
- Look at your data before training on it.

### ✔️ Check yourself

Commit to an answer (say it out loud or write it down) *before* you expand it.

**1.** A dataset has 1,000 examples and you use `batch_size=64`. How many batches per epoch, and how large is the last one?

<details>
<summary>Answer</summary>

`ceil(1000 / 64) = 16` batches; the last has `1000 − 15·64 = 40` examples (unless `drop_last=True`, which gives 15 batches).

</details>

**2.** Why should the validation loader use `shuffle=False`?

<details>
<summary>Answer</summary>

Shuffling does not change the evaluation result, so it only costs time, and a fixed order makes runs easier to compare and debug (for example, finding the same misclassified examples).

</details>

**3.** A teammate standardises the whole dataset, then splits it into train and test. Why is this a problem, even though the model never trains on test examples?

<details>
<summary>Answer</summary>

The standardisation statistics were computed using test examples, so information about the test set leaks into training. Test results become optimistically biased. The effect is small for mean/std on big datasets, but the same mistake with more powerful preprocessing (feature selection, imputation) can be large.

</details>

---
# Chapter 7. Project: an MLP on Fashion-MNIST, end to end

🎯 **In this chapter you will:** write reusable training and evaluation functions; compare against baselines; train an MLP; select the best epoch with the validation set; evaluate honestly on the test set; analyse errors with a confusion matrix; and save the model and use it for prediction.

This chapter puts everything together in the order you would follow at work:

1. **Data**: loaders for training, validation and test.
2. **Baselines**: what score does "no learning" or a linear model get?
3. **Training machinery**: `train_one_epoch`, `evaluate` and `fit`.
4. **Train** the MLP and read the learning curves.
5. **Select** the best epoch using validation accuracy, then **test once**.
6. **Analyse** the errors.
7. **Save** the model and use it for prediction.

If you start a **fresh session** here (for example after Colab disconnected), run the setup cell of Chapter 0 and then the hidden restore-point cell below. It re-creates what Chapters 4 and 6 defined. Otherwise you can skip it; it does nothing when those objects already exist.

In [ ]:
# @title 🔁 Restore point for Chapter 7 (needed only if you start a fresh session here) { display-mode: "form" }
if "device" not in globals():
    raise RuntimeError("Run the Chapter 0 setup cells first, then this cell.")
if "fm_transform" not in globals():
    from torch.utils.data import Dataset, TensorDataset, DataLoader, Subset
    from torchvision import datasets
    from torchvision.transforms import v2 as T

    def count_params(m):
        return sum(p.numel() for p in m.parameters() if p.requires_grad)

    raw_train = datasets.FashionMNIST(DATA_DIR, train=True, download=True)
    CLASSES = raw_train.classes
    perm = torch.randperm(len(raw_train), generator=torch.Generator().manual_seed(0))
    fm_val_idx, fm_train_idx = perm[:5000], perm[5000:]
    train_pixels = raw_train.data[fm_train_idx].float() / 255
    FM_MEAN, FM_STD = train_pixels.mean().item(), train_pixels.std().item()
    fm_transform = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True), T.Normalize(mean=[FM_MEAN], std=[FM_STD])])
    print("Restored the tools defined in earlier chapters.")

## 7.1 Data loaders
We reuse the split and the transform from §6.4: 55,000 training, 5,000 validation and 10,000 test images, normalised with training-split statistics.

In [ ]:
import copy, sys

NUM_WORKERS = 2 if sys.platform.startswith("linux") else 0   # see the note below
PIN = device.type == "cuda"

full_train = datasets.FashionMNIST(DATA_DIR, train=True, transform=fm_transform)
test_set = datasets.FashionMNIST(DATA_DIR, train=False, download=True, transform=fm_transform)
fm_train = maybe_subset(Subset(full_train, fm_train_idx.tolist()))     # same indices as §6.4
fm_val = maybe_subset(Subset(full_train, fm_val_idx.tolist()))
fm_test = maybe_subset(test_set)

train_loader = DataLoader(fm_train, batch_size=128, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN)
val_loader = DataLoader(fm_val, batch_size=512, num_workers=NUM_WORKERS, pin_memory=PIN)
test_loader = DataLoader(fm_test, batch_size=512, num_workers=NUM_WORKERS, pin_memory=PIN)
print(f"train {len(fm_train):,} | val {len(fm_val):,} | test {len(fm_test):,} | batches per epoch {len(train_loader)}")

**About `num_workers`.** With worker processes, the next batches are prepared on other CPU cores while the GPU trains on the current one. Colab has 2 CPU cores, hence 2 workers. On Windows and macOS, worker processes start differently, and a `Dataset` class defined inside a notebook can make them crash. That is why we use 0 workers there. In a `.py` script this restriction does not apply.

## 7.2 The training and evaluation functions
These two functions are the core of the project. Read every line: they combine everything from Chapters 2–6. You will reuse them, extended, in Notebook 2.

In [ ]:
def train_one_epoch(model, loader, loss_fn, optimizer):
    model.train()                                   # Dropout/BatchNorm in training mode
    total_loss, correct, n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)           # move each batch to the model's device
        logits = model(x)                           # 1. forward
        loss = loss_fn(logits, y)                   # 2. loss (the MEAN over this batch)
        optimizer.zero_grad()                       # 3. clear old gradients
        loss.backward()                             # 4. backward
        optimizer.step()                            # 5. update
        total_loss += loss.item() * len(y)          # undo the mean: the last batch may be smaller
        correct += (logits.argmax(dim=1) == y).sum().item()
        n += len(y)
    return total_loss / n, correct / n


@torch.no_grad()                                    # decorator: the whole function runs without autograd
def evaluate(model, loader, loss_fn):
    model.eval()                                    # Dropout off, BatchNorm uses running statistics
    total_loss, correct, n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        total_loss += loss_fn(logits, y).item() * len(y)
        correct += (logits.argmax(dim=1) == y).sum().item()
        n += len(y)
    return total_loss / n, correct / n

Three details deserve attention:

- **`loss.item() * len(y)`**: the loss is a mean over the batch. Multiplying by the batch size and dividing by `n` at the end gives the exact mean over the epoch, even though the last batch is smaller.
- **The training accuracy is approximate.** It is measured while the weights change during the epoch, and with Dropout active. For a clean number, call `evaluate` on the training set. The validation numbers come from `evaluate` and are exact.
- **`model.train()` is the first line of `train_one_epoch`.** Because `evaluate` switches the model to `eval()` mode, the training function must switch it back every time. Putting the switches inside the functions makes the "forgot to call `train()`" bug impossible.

`fit` runs the epochs, records a **history** for plotting, and keeps a copy of the weights from the epoch with the **best validation accuracy**:

In [ ]:
def fit(model, train_loader, val_loader, loss_fn, optimizer, epochs):
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_acc, best_state = -1.0, None
    for epoch in range(1, epochs + 1):
        t0 = time.perf_counter()
        tr_loss, tr_acc = train_one_epoch(model, train_loader, loss_fn, optimizer)
        va_loss, va_acc = evaluate(model, val_loader, loss_fn)
        for key, value in zip(history, (tr_loss, tr_acc, va_loss, va_acc)):
            history[key].append(value)
        if va_acc > best_acc:                       # model selection with the VALIDATION set
            best_acc = va_acc
            best_state = copy.deepcopy(model.state_dict())   # a real copy, not a reference
        print(f"epoch {epoch:2d} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
              f"val loss {va_loss:.4f} acc {va_acc:.4f} | {time.perf_counter() - t0:4.1f}s")
    model.load_state_dict(best_state)               # end with the best weights, not the last ones
    print(f"best validation accuracy: {best_acc:.4f}")
    return history

⚠️ **Why `copy.deepcopy`?** `model.state_dict()` returns references to the live parameter tensors. Without a copy, `best_state` would keep changing as training continues, and you would "restore" the final weights instead of the best ones. No error would tell you.

## 7.3 Baselines first
A number such as "85% accuracy" means nothing on its own. Before training the real model, measure what you get **without learning** and with the **simplest reasonable model**. If the real model cannot clearly beat the simple one, something is wrong.

In [ ]:
loss_fn = nn.CrossEntropyLoss()

untrained = nn.Sequential(nn.Flatten(), nn.Linear(784, 10)).to(device)
print("untrained model (chance level):", evaluate(untrained, val_loader, loss_fn))
print(f"  expected loss at chance: ln(10) = {math.log(10):.3f}; expected accuracy about 0.10")

set_seed(0)
linear = nn.Sequential(nn.Flatten(), nn.Linear(784, 10)).to(device)   # softmax regression
print("\nTraining a linear model (softmax regression):")
hist_linear = fit(linear, train_loader, val_loader, loss_fn,
                  torch.optim.AdamW(linear.parameters(), lr=1e-3), epochs=n_epochs(5))

A useful sanity check: **the initial loss of a classifier should be close to $\ln C$** (here $\ln 10 \approx 2.30$), because an untrained model spreads its probability roughly evenly across classes. If your initial loss is far above that, something is broken before training even starts.

The linear model typically reaches about 84% validation accuracy. That is the bar the MLP has to clear.

## 7.4 Training the MLP
Architecture: 784 → 256 → 128 → 10 with ReLU and Dropout 0.2. Optimiser: `AdamW` with learning rate 1e-3, a robust default (Notebook 2 explains optimiser choices). Expect roughly 5–10 seconds per epoch on a Colab GPU and a little longer on a CPU.

In [ ]:
set_seed(0)
mlp = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 256), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(128, 10),
).to(device)
print(f"parameters: {count_params(mlp):,}")
optimizer = torch.optim.AdamW(mlp.parameters(), lr=1e-3)
hist = fit(mlp, train_loader, val_loader, loss_fn, optimizer, epochs=n_epochs(15))

In [ ]:
def plot_history(hist, title=""):
    epochs = range(1, len(hist["train_loss"]) + 1)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.5))
    a1.plot(epochs, hist["train_loss"], "o-", label="train"); a1.plot(epochs, hist["val_loss"], "o-", label="validation")
    a1.set_xlabel("epoch"); a1.set_ylabel("loss"); a1.legend(); a1.set_title("Loss")
    a2.plot(epochs, hist["train_acc"], "o-", label="train"); a2.plot(epochs, hist["val_acc"], "o-", label="validation")
    a2.set_xlabel("epoch"); a2.set_ylabel("accuracy"); a2.legend(); a2.set_title("Accuracy")
    fig.suptitle(title); plt.tight_layout(); plt.show()

plot_history(hist, "MLP on Fashion-MNIST")

**Reading the curves.** Typically the training loss keeps falling throughout, while the validation loss falls for the first several epochs and then flattens or begins to creep up. Validation accuracy levels off around 88–89%. The growing gap between training and validation loss is the beginning of **overfitting**: the model starts memorising details of the training images that do not generalise.

That is exactly why `fit` keeps the best epoch instead of the last one. Be aware that the best validation accuracy is slightly **optimistic**: it is the maximum of several noisy measurements on the same 5,000 images, so it partly reflects luck. That is one more reason to keep an untouched test set. Notebook 2 turns this into **early stopping**, which also stops training once validation stops improving and saves the remaining time.

Your exact numbers will differ slightly from the ones quoted in the text, because of random initialisation and data order, and because different hardware rounds differently. The pattern is what matters.

## 7.5 The final, honest evaluation on the test set
We chose the architecture, the learning rate and the epoch using the **validation** set only. Now, and only now, we evaluate on the test set, **once**. If you go back and change the model because of this number, the test set stops being an unbiased estimate.

In [ ]:
test_loss, test_acc = evaluate(mlp, test_loader, loss_fn)
print(f"MLP test accuracy:    {test_acc:.4f}  (test loss {test_loss:.4f})")
print(f"linear test accuracy: {evaluate(linear, test_loader, loss_fn)[1]:.4f}")
record("linear model, test accuracy", evaluate(linear, test_loader, loss_fn)[1])
record("MLP, test accuracy", test_acc)

## 7.6 Which mistakes does the model make? The confusion matrix
A single accuracy hides **where** the errors are. The **confusion matrix** `C` counts, for every true class `i` and predicted class `j`, how many test images of class `i` were predicted as `j`. The diagonal holds the correct predictions.

The cell below builds it with one tensor trick: every (true, predicted) pair is turned into one index `true · 10 + predicted` between 0 and 99, `torch.bincount` counts how often each index occurs, and a reshape turns the 100 counts into a 10×10 matrix.

In [ ]:
@torch.no_grad()
def predict_all(model, loader):
    model.eval()
    preds, targets = [], []
    for x, y in loader:
        preds.append(model(x.to(device)).argmax(dim=1).cpu())
        targets.append(y)
    return torch.cat(preds), torch.cat(targets)

pred, true = predict_all(mlp, test_loader)
cm = torch.bincount(true * 10 + pred, minlength=100).reshape(10, 10)

plt.figure(figsize=(7, 6))
plt.imshow(cm, cmap="Blues")
for i in range(10):
    for j in range(10):
        plt.text(j, i, cm[i, j].item(), ha="center", va="center", fontsize=7,
                 color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.xticks(range(10), CLASSES, rotation=60, fontsize=8); plt.yticks(range(10), CLASSES, fontsize=8)
plt.xlabel("predicted"); plt.ylabel("true"); plt.title("Confusion matrix (test set)")
plt.tight_layout(); plt.show()

per_class = cm.diag() / cm.sum(dim=1)
for i in per_class.argsort():
    print(f"{CLASSES[i]:12s} accuracy {per_class[i]:.3f}")

The pattern is usually very clear. Trousers, bags and footwear are easy. **Shirt** is by far the hardest class, and it is confused mostly with T-shirt/top, pullover and coat. These are visually similar upper-body garments, and at 28×28 pixels even people disagree about some of them.

This is the kind of analysis that tells you what to do next. More epochs of the same MLP will not fix shirt-versus-coat confusion. A model that understands local shapes and textures, a **convolutional network** (Notebook 2), will.

Look at some of the mistakes themselves:

In [ ]:
wrong = (pred != true).nonzero().squeeze(1)[:12]
fig, axes = plt.subplots(2, 6, figsize=(12, 4.4))
for ax, i in zip(axes.flat, wrong.tolist()):
    img, _ = test_set.data[i], test_set.targets[i]
    ax.imshow(img, cmap="gray"); ax.axis("off")
    ax.set_title(f"true: {CLASSES[true[i]]}\npred: {CLASSES[pred[i]]}", fontsize=8)
plt.suptitle("Misclassified test images"); plt.tight_layout(); plt.show()

## 7.7 Using the model: predicting one new image
In deployment, inputs arrive one at a time and must go through **exactly the same preprocessing** as during training. The recipe has four steps, and forgetting any of them is a classic bug:

1. the same transform (including the same normalisation statistics);
2. a batch dimension, because models always expect one;
3. `eval()` mode and `no_grad()`;
4. softmax only now, because we want to report probabilities.

In [ ]:
@torch.no_grad()
def predict_image(model, pil_image):
    model.eval()                                        # 3.
    x = fm_transform(pil_image)                         # 1. (1, 28, 28), same preprocessing as training
    x = x.unsqueeze(0).to(device)                       # 2. (1, 1, 28, 28)
    probs = torch.softmax(model(x), dim=1)[0]           # 4. (10,)
    top = probs.argmax().item()
    return CLASSES[top], probs[top].item()

raw_test = datasets.FashionMNIST(DATA_DIR, train=False)   # the raw PIL images
for i in range(5):
    img, label = raw_test[i]
    name, p = predict_image(mlp, img)
    print(f"true: {CLASSES[label]:12s} predicted: {name:12s} (probability {p:.2f})")

## 7.8 Saving the model for later
Save the weights **together with what is needed to rebuild and use the model**: the architecture settings, the class names and the normalisation statistics. A weights file without them is hard to use six months later.

In [ ]:
def build_mlp(hidden=(256, 128), p_drop=0.2):
    layers, d = [nn.Flatten()], 784
    for h in hidden:
        layers += [nn.Linear(d, h), nn.ReLU(), nn.Dropout(p_drop)]
        d = h
    return nn.Sequential(*layers, nn.Linear(d, 10))

bundle = {
    "state_dict": mlp.state_dict(),
    "config": {"hidden": (256, 128), "p_drop": 0.2},
    "classes": CLASSES,
    "normalization": {"mean": FM_MEAN, "std": FM_STD},
    "test_accuracy": test_acc,
}
torch.save(bundle, DATA_DIR / "fashion_mlp.pt")

# Later, possibly in another program:
loaded = torch.load(DATA_DIR / "fashion_mlp.pt", map_location="cpu")
model2 = build_mlp(**loaded["config"])
model2.load_state_dict(loaded["state_dict"])
model2.to(device)
print("reloaded model, test accuracy:", evaluate(model2, test_loader, loss_fn)[1], "| stored:", loaded["test_accuracy"])

## 7.9 When training does not work: the first two checks
Neural-network bugs rarely crash. Usually the loss just does not go down as it should. Two quick checks catch most problems before you waste hours:

1. **Check the initial loss.** It should be about $\ln C$ (§7.3).
2. **Overfit one small batch.** A correct model and training loop can memorise 64 examples almost perfectly within a few hundred steps. If the loss does not go to nearly zero, the bug is in the model, the loss or the loop, not in the amount of data or the hyper-parameters.

In [ ]:
set_seed(0)
xb, yb = next(iter(train_loader))
xb, yb = xb[:64].to(device), yb[:64].to(device)
probe = build_mlp(p_drop=0.0).to(device)                # no dropout: we WANT to memorise
opt = torch.optim.AdamW(probe.parameters(), lr=1e-3)
for step in range(201):
    loss = loss_fn(probe(xb), yb)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 50 == 0:
        print(f"step {step:3d}  loss {loss.item():.4f}")

| Symptom | Most likely causes |
| --- | --- |
| Initial loss far above $\ln C$ | Inputs not normalised; a bad initialisation; labels in the wrong format |
| Loss stays exactly flat | `optimizer.step()` missing or before `backward()`; parameters not registered (§4.3); learning rate 0 |
| Loss goes to `nan` | Learning rate too high; a `log(0)` or division by zero in a hand-written loss |
| Loss falls, but the model cannot overfit one batch | A softmax before `CrossEntropyLoss`; a shape bug such as `(N, N)` broadcasting |
| Validation accuracy noisy between identical evaluations | `model.eval()` forgotten |
| Great validation, poor in production | Different preprocessing at prediction time (§7.7) |

Notebook 2 has a full chapter on systematic debugging.

## 🧪 Exercises

**Exercise 7.1: capacity and regularisation.** `★★☆ · ~30 min (mostly waiting)`
Use `build_mlp` and `fit` to train three variants for 10 epochs each, and fill in a table of **best validation accuracy** and the **final gap between training and validation loss**:
(a) `hidden=(64,)`, no dropout; (b) `hidden=(512, 256)`, no dropout; (c) `hidden=(512, 256)`, `p_drop=0.3`.
Which variant overfits most? Which generalises best? Important: compare them on **validation** accuracy, never on the test set.

In [ ]:
# ✍️ Your attempt here.

In [ ]:
# @title ✅ Solution (double-click to reveal) { display-mode: "form" }
# Solutions only execute when RUN_SOLUTIONS = True (setup cell), so "Run all" stays fast.
if RUN_SOLUTIONS:
    results = {}
    for name, cfg in {"(a) small": dict(hidden=(64,), p_drop=0.0),
                      "(b) big": dict(hidden=(512, 256), p_drop=0.0),
                      "(c) big + dropout": dict(hidden=(512, 256), p_drop=0.3)}.items():
        set_seed(0)
        m = build_mlp(**cfg).to(device)
        print(f"--- {name}")
        h = fit(m, train_loader, val_loader, loss_fn, torch.optim.AdamW(m.parameters(), lr=1e-3), n_epochs(10))
        results[name] = (max(h["val_acc"]), h["val_loss"][-1] - h["train_loss"][-1])
    for name, (acc, gap) in results.items():
        print(f"{name:20s} best val acc {acc:.4f} | final val-train loss gap {gap:+.3f}")
    # Typically: (b) has the largest gap (overfitting: high variance); (a) has a small gap but lower
    # accuracy (limited capacity: higher bias); (c) narrows the gap of (b) and usually matches or beats
    # its best validation accuracy. Capacity and regularisation are tuned together.
else:
    print("Solution not executed (RUN_SOLUTIONS = False).")

**Exercise 7.2: SGD versus AdamW.** `★★☆ · ~20 min`
Train `build_mlp()` for 10 epochs with (a) `torch.optim.SGD(lr=0.1)`, (b) `torch.optim.SGD(lr=0.05, momentum=0.9)`, and (c) `torch.optim.AdamW(lr=1e-3)`. Plot the three validation-accuracy curves on one axis. Which converges fastest? (Notebook 2 explains momentum and Adam in detail.)

In [ ]:
# ✍️ Your attempt here.

In [ ]:
# @title ✅ Solution (double-click to reveal) { display-mode: "form" }
# Solutions only execute when RUN_SOLUTIONS = True (setup cell), so "Run all" stays fast.
if RUN_SOLUTIONS:
    curves = {}
    for name, make_opt in {"SGD lr=0.1": lambda p: torch.optim.SGD(p, lr=0.1),
                           "SGD+momentum lr=0.05": lambda p: torch.optim.SGD(p, lr=0.05, momentum=0.9),
                           "AdamW lr=1e-3": lambda p: torch.optim.AdamW(p, lr=1e-3)}.items():
        set_seed(0)
        m = build_mlp().to(device)
        print(f"--- {name}")
        curves[name] = fit(m, train_loader, val_loader, loss_fn, make_opt(m.parameters()), n_epochs(10))["val_acc"]
    plt.figure(figsize=(6, 3.5))
    for name, c in curves.items():
        plt.plot(range(1, len(c) + 1), c, "o-", label=name)
    plt.xlabel("epoch"); plt.ylabel("validation accuracy"); plt.legend(); plt.show()
    # With these settings all three end up close; AdamW and SGD+momentum usually get there in
    # fewer epochs than plain SGD. The ranking can change with the learning rate, which is why
    # optimizers are compared at their own tuned learning rates, never at one shared value.
else:
    print("Solution not executed (RUN_SOLUTIONS = False).")

**Exercise 7.3: a new dataset in one line.** `★☆☆ · ~10 min`
Replace `datasets.FashionMNIST` with `datasets.MNIST` (handwritten digits) in a copy of the loader code of §7.1. Following the rule of §6.2, **recompute** the normalisation statistics on MNIST's own training split. Then train `build_mlp()` for 5 epochs. What test accuracy do you get, and why is it higher than on Fashion-MNIST?

In [ ]:
# ✍️ Your attempt here.

In [ ]:
# @title ✅ Solution (double-click to reveal) { display-mode: "form" }
# Solutions only execute when RUN_SOLUTIONS = True (setup cell), so "Run all" stays fast.
if RUN_SOLUTIONS:
    mn_raw = datasets.MNIST(DATA_DIR, train=True, download=True)
    mn_pixels = mn_raw.data[fm_train_idx].float() / 255                     # MNIST's training split only
    mn_mean, mn_std = mn_pixels.mean().item(), mn_pixels.std().item()
    print(f"MNIST statistics: mean {mn_mean:.4f}, std {mn_std:.4f}  (Fashion-MNIST: {FM_MEAN:.4f}, {FM_STD:.4f})")
    mn_tf = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True), T.Normalize([mn_mean], [mn_std])])
    mn_train_full = datasets.MNIST(DATA_DIR, train=True, transform=mn_tf)
    mn_test = maybe_subset(datasets.MNIST(DATA_DIR, train=False, download=True, transform=mn_tf))
    mn_train = maybe_subset(Subset(mn_train_full, fm_train_idx.tolist()))
    mn_val = maybe_subset(Subset(mn_train_full, fm_val_idx.tolist()))
    set_seed(0)
    m = build_mlp().to(device)
    fit(m, DataLoader(mn_train, batch_size=128, shuffle=True, num_workers=NUM_WORKERS),
        DataLoader(mn_val, batch_size=512, num_workers=NUM_WORKERS),
        loss_fn, torch.optim.AdamW(m.parameters(), lr=1e-3), n_epochs(5))
    print("MNIST test accuracy:", evaluate(m, DataLoader(mn_test, batch_size=512), loss_fn)[1])
    # Typically about 98%: digits are separated by simple stroke patterns, while several Fashion-MNIST
    # classes (shirt, T-shirt, pullover, coat) differ only in subtle details. Nothing else in the
    # pipeline had to change, which is the point of separating data, model and training loop.
else:
    print("Solution not executed (RUN_SOLUTIONS = False).")

### 📌 Key takeaways
- Put `model.train()` / `model.eval()` (and `no_grad`) **inside** your training and evaluation functions.
- Compare against baselines. Check that the initial loss is about $\ln C$, and that the model can overfit one batch.
- Select epochs and hyper-parameters on the validation set; touch the test set once.
- `copy.deepcopy(model.state_dict())` to keep the best weights.
- At prediction time: same preprocessing, a batch dimension, `eval()`, `no_grad()`.

### ✔️ Check yourself

Commit to an answer (say it out loud or write it down) *before* you expand it.

**1.** Why does `train_one_epoch` multiply `loss.item()` by `len(y)`?

<details>
<summary>Answer</summary>

The loss is the mean over the batch. Summing `mean × batch size` and dividing by the total count gives the exact mean over the epoch, even when the last batch is smaller.

</details>

**2.** You keep `best_state = model.state_dict()` (no copy). What goes wrong?

<details>
<summary>Answer</summary>

The dictionary holds references to the live parameters, so it changes as training continues. At the end you would 'restore' the final weights, not the best ones.

</details>

**3.** After looking at the test accuracy, you add dropout and retrain, and the test accuracy improves. Can you report the new number as an unbiased estimate?

<details>
<summary>Answer</summary>

No. The test set influenced a modelling decision, so it has effectively become a validation set. Use the validation set for such decisions, or hold out a fresh test set.

</details>

**4.** Your model predicts well in the notebook but badly on images sent by a web app. Name the first thing you would check.

<details>
<summary>Answer</summary>

That the app applies exactly the same preprocessing: the same scaling to [0, 1], the same normalisation statistics, the same image size and channel layout, plus a batch dimension and `eval()` mode.

</details>

---
# Summary: the complete template
This is the skeleton of nearly every supervised PyTorch project. Everything in Notebook 2 extends one of its parts.

```python
# ---- data ----
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=512)

# ---- model, loss, optimizer ----
model = MyModel().to(device)                     # outputs LOGITS
loss_fn = nn.CrossEntropyLoss()                  # int64 targets of shape (N,)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

# ---- training ----
for epoch in range(epochs):
    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        loss = loss_fn(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        ...                                      # validation metrics; keep the best weights

# ---- once, at the end ----
test_metrics = evaluate(model, test_loader, loss_fn)
torch.save({"state_dict": model.state_dict(), "config": cfg}, "model.pt")
```

## Your results
The cell below prints the numbers **you** measured in this notebook. Values quoted in the text (such as "about 89%") are typical outcomes, not guarantees; your own run is the ground truth.

In [ ]:
for name, value in RESULTS.items():
    print(f"{name:35s} {value:.4f}")

## What comes next: Notebook 2
You can now build and train a model correctly. **Notebook 2 (PyTorch in Practice)** teaches the tools that make you effective at work and let you read research code:

- how tensors are stored in memory (strides, views, `contiguous`), and `einsum` for writing formulas from papers directly;
- optimizers, learning-rate schedules, gradient clipping and early stopping;
- checkpoints that resume training exactly, and reproducibility;
- weight averaging (EMA and SWA);
- convolutional networks, built from first principles;
- transfer learning: freezing layers and fine-tuning pretrained models;
- mixed precision and `torch.compile`, debugging, and exporting models for production.